# LEAN-LLM-OPT (gpt-oss-20b) component ablation: RAG Only

This notebook runs the gpt-oss-20b workflow on the 101 Large-scale-or instances with one component setting. RAG Only removes the worked formulation and code examples and keeps the data-handling tools.

1. Start a local Ollama service with `gpt-oss:20b` and `nomic-embed-text`, and configure a Gurobi license.
2. Run the definition cells from top to bottom.
3. Enable the run cell at the end. Results are written to `outputs/ablation_gpt_oss_20b_rag_only/automatic`.

## 1. Imports and configuration

Configure the selected model service, embeddings, project paths, and Gurobi license here.


In [ ]:
from __future__ import annotations
import ast
import fcntl
import gurobipy as gp
import hashlib
import json
import math
import numpy as np
import os
import openai
import pandas as pd
import pprint
import re
import tempfile
from IPython.display import display
from dotenv import load_dotenv
from functools import lru_cache
from langchain_classic.agents import AgentType, Tool, initialize_agent
from langchain_classic.chains import LLMChain, RetrievalQA
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pathlib import Path
from threading import RLock
from typing import Any
from urllib.parse import quote

import httpx

import requests
from langchain_community.chat_models import ChatOllama
from langchain_core.embeddings import Embeddings

from langchain_core.agents import AgentAction


In [ ]:
PROJECT_ROOT = Path(os.environ.get("LEAN_PROJECT_ROOT", os.environ.get("LEAN_LLM_OPT_ROOT", Path.cwd()))).expanduser().resolve()
BENCHMARK_PATH = PROJECT_ROOT / "Test_Dataset/Large-scale-or/Large-scale-or-101.csv"
OTHER_BENCHMARK_PATH = PROJECT_ROOT / "Test_Dataset/Small-scale/IndustryOR_fixedV2.csv"
NOTEBOOK_FILENAME = 'Ablation_Study_gpt_oss_20b_Large-scale_RAG_Only.ipynb'
RESULTS_DIR = Path(os.environ.get("LEAN_RESULTS_DIR", PROJECT_ROOT / "outputs" / "ablation_gpt_oss_20b_rag_only"))
NOTEBOOK_PATH = Path(os.environ.get("LEAN_NOTEBOOK_PATH", PROJECT_ROOT / NOTEBOOK_FILENAME)).resolve()
load_dotenv(PROJECT_ROOT / ".env.local", override=False)
load_dotenv(PROJECT_ROOT / ".env", override=False)
user_api_key = os.environ.get("OPENAI_API_KEY")
VERSION = "gpt-oss-20b-ablation-rag-only"
MODEL_PROVIDER = 'ollama'
MODEL_SNAPSHOT = 'gpt-oss:20b'
EMBEDDING_MODEL = 'nomic-embed-text'
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434").rstrip("/")
CLASS_LABELS = ["TP", "NRM", "RA", "FLP", "AP", "Mixture", "Others"]
WORKFLOW_ROUTES = ["TP", "NRM", "FLP", "AP", "RA", "Others"]
CSVQA_MODE_BY_ROUTE = {"NRM": "planned", "RA": "planned", "TP": "planned", "AP": "planned", "FLP": "planned", "Others": "planned"}
RAG_EXAMPLES_ALL_PATH = "Large_Scale_Or_Files/RAG_Examples_All.csv"
CACHE_SCHEMA_VERSION = VERSION
API_MAX_RETRIES = 2
NRM_RETRY_ON_TRUNCATION = False


## 2. Shared model clients

All calls use gpt-oss:20b through a local Ollama service (temperature 0, seed 42, 32,768-token context) and nomic-embed-text embeddings.


In [ ]:
HTTP_RETRY_EVENTS = []

def _record_http_request(request):
    retry_index = int(request.headers.get("x-stainless-retry-count", "0"))
    if retry_index:
        HTTP_RETRY_EVENTS.append({"endpoint": request.url.path, "retry_index": retry_index})

@lru_cache(maxsize=1)
def _get_http_client():
    return httpx.Client(timeout=180, event_hooks={"request": [_record_http_request]})

class ModelOutputTruncated(RuntimeError):
    """The model service explicitly reported an incomplete response."""

class RejectTruncatedOutput(BaseCallbackHandler):
    raise_error = True

    def on_llm_end(self, response, **kwargs):
        for choices in response.generations:
            for generation in choices:
                metadata = getattr(getattr(generation, "message", None), "response_metadata", {}) or {}
                info = generation.generation_info or {}
                reason = str(info.get("finish_reason") or info.get("done_reason")
                             or metadata.get("finish_reason") or metadata.get("done_reason") or "").lower()
                if reason in {"length", "max_tokens", "max_output_tokens"}:
                    raise ModelOutputTruncated(f"Model output is incomplete: finish_reason={reason}")

class OllamaEmbeddingAdapter(Embeddings):
    def __init__(self, model, base_url, timeout=500):
        self.model, self.base_url, self.timeout = model, base_url.rstrip("/"), timeout

    def _embed(self, texts):
        values = [str(value) for value in texts]
        response = requests.post(self.base_url + "/api/embed",
                                 json={"model": self.model, "input": values, "truncate": True},
                                 timeout=self.timeout)
        response.raise_for_status()
        vectors = response.json().get("embeddings")
        if not isinstance(vectors, list) or len(vectors) != len(values):
            raise RuntimeError("Ollama returned an invalid embedding batch")
        return vectors

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed([text])[0]

class SeededChatOllama(ChatOllama):
    @property
    def _default_params(self):
        parameters = dict(super()._default_params)
        parameters["options"] = dict(parameters.get("options") or {})
        parameters["options"]["seed"] = int(os.environ.get("LEAN_SEED", "42"))
        parameters["think"] = os.environ.get("LEAN_REASONING_EFFORT", "medium")
        return parameters

def require_model_backend():
    """Validate provider configuration without sending a generation request."""
    if MODEL_PROVIDER == "ollama":
        if not OLLAMA_BASE_URL.startswith(("http://", "https://")):
            raise ValueError("OLLAMA_BASE_URL must identify an Ollama service")
        return OLLAMA_BASE_URL
    if MODEL_PROVIDER != "openai":
        raise ValueError(f"Unsupported model provider: {MODEL_PROVIDER}")
    load_dotenv(PROJECT_ROOT / ".env.local", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
    key = (os.environ.get("OPENAI_API_KEY") or user_api_key or "").strip()
    if not key:
        raise ValueError("Set OPENAI_API_KEY before using the OpenAI provider")
    return key

def make_llm(*, timeout=500):
    require_model_backend()
    if MODEL_PROVIDER == "ollama":
        return SeededChatOllama(model=MODEL_SNAPSHOT, base_url=OLLAMA_BASE_URL, temperature=0.0,
                               num_ctx=int(os.environ.get("LEAN_NUM_CTX", "32768")),
                               num_predict=int(os.environ.get("LEAN_NUM_PREDICT", "12288")),
                               timeout=timeout, callbacks=[RejectTruncatedOutput()])
    return ChatOpenAI(model_name=MODEL_SNAPSHOT, openai_api_key=require_model_backend(),
                      temperature=0.0, top_p=1, n=1, max_retries=API_MAX_RETRIES,
                      http_client=_get_http_client(), timeout=timeout, callbacks=[RejectTruncatedOutput()])

def make_embeddings():
    if MODEL_PROVIDER == "ollama":
        return OllamaEmbeddingAdapter(EMBEDDING_MODEL, OLLAMA_BASE_URL)
    return OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=require_model_backend(),
                            max_retries=API_MAX_RETRIES, http_client=_get_http_client())

MODEL_INTERFACE_INSTRUCTIONS = """
Mandatory execution interface, independent of every retrieved example:
Create exactly one Gurobi model and expose the solved, live model at module scope as m.
Preferred function pattern (replace the placeholder with the actual modeling code):
    def solve_problem(...):
        m = gp.Model(...)
        # define variables, objective, and all constraints
        m.optimize()
        return m
    m = solve_problem(...)
A direct global script is also allowed, but its final m must be the valid Gurobi model.
If a function creates the model, explicitly return m and call that function at module scope.
Defining a function without calling it is incomplete. Reserve m and model for Gurobi models.
Do not assign the return of optimize() to m/model: never write m = m.optimize().
Do not close or dispose the model, or exit a model context manager, before the outer harness
extracts results. Do not catch and hide execution errors. The harness checks the actual solver
status and extracts objective and variable values; printed values cannot replace this interface.
"""


## 3. CSV loading and route modes

CSV-containing cases use a common declarative planner and source-data extraction. Each route selects its reference examples; current data and query restrictions are preserved.


In [ ]:
def normalize_route(value: Any) -> str:
    text = str(value or "").strip().upper()
    aliases = {
        "NETWORK REVENUE MANAGEMENT": "NRM", "NETWORK REVENUE MANAGEMENT PROBLEM": "NRM",
        "RESOURCE ALLOCATION": "RA", "RESOURCE ALLOCATION PROBLEM": "RA",
        "TRANSPORTATION": "TP", "TRANSPORTATION PROBLEM": "TP",
        "ASSIGNMENT": "AP", "ASSIGNMENT PROBLEM": "AP",
        "FACILITY LOCATION": "FLP", "FACILITY LOCATION PROBLEM": "FLP",
    }
    text = aliases.get(text, text)
    if text in {"MIXTURE", "OTHER", "OTHERS"} or text.startswith("OTHERS"):
        return "Others"
    if text == "UFLP":
        return "FLP"
    if text in {"NRM", "RA", "TP", "AP", "FLP"}:
        return text
    raise ValueError(f"Unsupported execution route: {value!r}")


FILTER_OPERATORS = {
    "exact", "prefix", "contains", "in", "not_in", "eq", "ne",
    "gt", "ge", "lt", "le", "between", "is_null", "not_null",
}


FILTER_DTYPES = {"string", "number", "date"}


CSVQA_LEGACY_SYSTEM_PROMPT = (
    "Get every CSV row needed by the query in source order. Preserve exact identifiers and "
    "numeric values. If the query names a subset, get that complete subset; otherwise get "
    "all data. Use only the supplied context: {context}"
)


CSVQA_LEGACY_TOOL_DESCRIPTION = (
    "Get complete source-ordered CSV evidence needed to formulate the optimization problem."
)

def invoke_react_with_required_csvqa(agent, query, route):
    """Validate tool use after one agent invocation; never correct or rerun it."""
    result = agent.invoke(query)
    calls = sum(isinstance(step, tuple) and len(step) >= 2
                and getattr(step[0], "tool", None) == "CSVQA"
                for step in result.get("intermediate_steps", []))
    if calls != 1:
        raise RuntimeError(f"{route} ReAct must call CSVQA exactly once; observed {calls}")
    return result


CSVQA_PLANNED_PROMPTS = {'NRM': 'Select every source row and column needed to define the revenue-management objective, decision entities, demand limits, inventory or capacity limits, and query-specific bounds. Infer exact columns from the query and schema. Filter only an explicit subset.'}

CSVQA_PLANNED_PROMPTS["RA"] = (
    "Select source columns for decision identifiers, objective coefficients, resource consumption, "
    "every applicable capacity dimension, and query-specific bounds. Keep required scalar parameter "
    "rows and resource tables even when products are filtered. Infer roles from the query and schema, "
    "never column position, numeric magnitude, or filename alone. Exclude administrative/redundant "
    "columns only when they have no role in the query. Preserve exact IDs and all required coefficients. "
    "Filter only subsets explicitly requested by the original query."
)

CSVQA_TOOL_DESCRIPTIONS = {'NRM': 'Return the canonical CSV rows, columns, filters, and relationships needed by the query.'}

CSVQA_TOOL_DESCRIPTIONS["RA"] = "Return source-derived RA records, columns, filters, and resource relationships."


def read_csv_compat(path, **kwargs):
    """Read a project-relative or absolute CSV with the requested pandas options."""
    path = Path(path).expanduser()
    path = path if path.is_absolute() else PROJECT_ROOT / path
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "gbk", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeError(f"Unable to decode CSV: {path}") from last_error


def _read_csv(path):
    """Read values as text and remove only empty structure or a clear export index."""
    frame = read_csv_compat(path, dtype=str, keep_default_na=False)
    frame.columns = [str(column).lstrip("\ufeff") for column in frame.columns]
    if frame.empty:
        return frame
    nonempty = frame.apply(lambda column: column.astype(str).str.strip().ne(""))
    frame = frame.loc[nonempty.any(axis=1), nonempty.any(axis=0)].copy()
    for column in list(frame.columns):
        values = frame[column].astype(str).tolist()
        sequential = values in (
            [str(i) for i in range(len(values))],
            [str(i) for i in range(1, len(values) + 1)],
        )
        if sequential and re.fullmatch(r"(?:unnamed(?::\s*\d+)?|index)", column.strip(), re.I):
            frame = frame.drop(columns=column)
    return frame


def _load_tables(dataset_address):
    """Resolve one-or-more newline-separated CSV paths in source order."""
    tables = []
    for raw in str(dataset_address or "").splitlines():
        raw = raw.strip()
        if not raw:
            continue
        path = Path(raw).expanduser()
        path = path if path.is_absolute() else (PROJECT_ROOT / path).resolve()
        if not path.is_file():
            raise FileNotFoundError(f"CSVQA cannot find dataset file: {path}")
        tables.append({"file_index": len(tables), "path": path, "frame": _read_csv(path)})
    if not tables:
        raise ValueError("CSVQA requires at least one CSV path")
    return tables


def _build_profile(tables, query):
    """Expose schema, compact samples, and evidence for query-named terms."""
    quote_pattern = r'"([^\"]+)"|\'([^\']+)\'|“([^”]+)”|‘([^’]+)’'
    terms = list(dict.fromkeys(
        next(value for value in match.groups() if value is not None).strip()
        for match in re.finditer(quote_pattern, str(query or ""))
    ))
    key = lambda value: "".join(ch for ch in str(value or "").casefold() if ch.isalnum())
    profile = []
    for table in tables:
        frame = table["frame"]
        term_matches = {}
        for term in terms:
            wanted, value_matches = term.casefold(), {}
            for column in frame.columns:
                values = frame[column].astype(str).str.strip().str.casefold()
                counts = {
                    "exact": int(values.eq(wanted).sum()),
                    "prefix": int(values.str.startswith(wanted).sum()),
                    "contains": int(values.str.contains(wanted, regex=False).sum()),
                }
                if counts["contains"]:
                    value_matches[str(column)] = counts
            term_matches[term] = {
                "column_matches": [str(column) for column in frame.columns if key(column) == key(term)],
                "value_matches": value_matches,
            }
        profile.append({
            "file_index": table["file_index"],
            "file_name": table["path"].name,
            "rows": len(frame),
            "columns": list(frame.columns),
            "sample_values": {
                str(column): frame[column].astype(str).loc[lambda values: values.str.strip().ne("")]
                .drop_duplicates().head(4).tolist()
                for column in frame.columns
            },
            "query_terms": term_matches,
        })
    return profile


for _route in ("TP", "AP", "FLP"):
    CSVQA_PLANNED_PROMPTS[_route] = (
        "Select every source row and column required by the original query, including entity IDs, "
        "matrix coefficients and axes, costs, capacities, demands and additional restrictions. "
        "Use exact source column names and retain business keys. Exclude a column only when it "
        "has no role in the query. Illustrative examples do not restrict the entity set."
    )
    CSVQA_TOOL_DESCRIPTIONS[_route] = "Return source-derived records, exact columns, filters and matrix axes."

ALIGNMENT_MODEL_HINT = '\nKeep the formulation concise and symbolic. Do not transcribe the full data payload,\nrepeat coefficients, reproduce examples, or enumerate all data rows in your answer.\nData Mapping must explicitly retain every restriction stated in the current query:\nname the filter column and condition and define the decision set over selected rows.\nDo not enlarge a category-restricted decision set to all categories.\nBefore finalizing, check that every variable index in every constraint belongs to\nits declared domain. A predecessor at the first modeled period must use the given\ninitial binary state or an explicitly provided continuous boundary, not an undefined\ndecision variable at period zero. Do not invent continuous pre-horizon quantities.\nFor a tour, keep the edge orientation and subtour formulation consistent. Directed\narcs have one incoming and one outgoing arc per node and may use directed MTZ order\nconstraints. Undirected edges have degree two and require an undirected subtour\nformulation; never apply both directions of directed MTZ to one undirected edge.\n'
for _route in CSVQA_PLANNED_PROMPTS:
    CSVQA_PLANNED_PROMPTS[_route] += "\n" + ALIGNMENT_MODEL_HINT

CSVQA_TOOL_DESCRIPTIONS["Others"] = "Return source-derived records, exact columns, filters and matrix axes."

CSVQA_PLANNED_PROMPTS["Others"] = 'Select all source records and columns needed for the current query, including decision entities, objective coefficients, constraint coefficients, bounds, scalar parameters, and matrix axes. Infer roles from the query and actual schema without assuming a predefined model type. Filter only an explicit query subset. Preserve source identifiers, zeros, matrix orientation, and structural blanks; do not solve or invent coefficients.'

for _route in CSVQA_PLANNED_PROMPTS:
    CSVQA_PLANNED_PROMPTS[_route] += '\nFor resource/capacity limits, select their per-unit consumption columns (weight, time, etc.) as well as capacities. Reference-example files and parameters are not current inputs.'


## 4. Planned CSV extraction

The model proposes columns and filters as JSON. Python applies the plan to the full CSV tables and preserves source identifiers and row order. These checks validate the data plan, not generated Python.


In [ ]:

def _relational_export_layout(tables):
    """Recognize a source schema, never a benchmark ID, filename or target value."""
    columns = [set(t["frame"].columns if "frame" in t else t.get("columns", []))
               for t in tables]
    return (any({"authorized", "minimum_lot", "maximum_order"} <= c for c in columns)
            and any({"item_ref", "prerequisite_ref"} <= c for c in columns)
            and any({"item_ref", "resource", "amount", "unit"} <= c for c in columns))

def _export_field_domains(table):
    """Bounded complete categorical evidence; not row selection or data repair."""
    fields = {"table", "unit", "currency", "action", "authorized"}
    columns = table["frame"].columns if "frame" in table else table.get("columns", [])
    domains = {}
    for column in columns:
        if str(column).casefold() not in fields:
            continue
        raw = (table["frame"][column].astype(str).tolist() if "frame" in table
               else [r.get("values", {}).get(column, "") for r in table.get("records", [])])
        values = list(dict.fromkeys(str(v) for v in raw if str(v).strip()))
        domains[column] = {"values": values[:32], "distinct_count": len(values),
                           "complete": len(values) <= 32}
    return domains if len(json.dumps(domains, ensure_ascii=False)) <= 2048 else {}


def _annotate_logical_tables(payload, source_tables=None):
    """Index actual source labels and exact table IDs; never select or alter a row."""
    if not _relational_export_layout(source_tables or payload.get("tables", [])):
        return payload
    index, evidence = {}, {}
    sources = {t["file_index"]: t for t in (source_tables or [])}
    for table in payload["tables"]:
        source = sources.get(table.get("file_index"))
        record_labels = {}
        for record in table["records"]:
            observed = record["values"].get("table")
            if source is not None and "table" in source["frame"].columns:
                label = str(source["frame"].loc[record["source_row"], "table"])
                if observed is not None and str(observed) != label:
                    raise ValueError("Returned/source table label identity mismatch")
            else:
                label = str(observed or "")
            if label:
                record_labels[str(record["source_row"])] = label
        labels = list(dict.fromkeys(record_labels.values()))
        for label in labels:
            index.setdefault(label, []).append(table["table_id"])
        evidence[table["table_id"]] = {
            "source_labels": labels, "columns": table["columns"],
            "returned_rows": len(table["records"]),
            "original_rows": table["original_rows"],
            "field_domains": _export_field_domains(table)}
        if len(labels) > 1:
            evidence[table["table_id"]]["record_labels"] = record_labels
    return {**payload, "logical_table_index": index, "logical_table_evidence": evidence}

INDEX_PLANNER_CONTRACT = """Source table labels are given in _source_field_domains. Map every physical
fragment needed by the query. Retain all fields, including the table label, eligibility,
units and event keys. Retain master attributes of ineligible choices; selection/quantity
must be disabled in the model, not by an early authorization filter. For event sources,
retain revisions and DELETE rows; extraction must not resolve snapshots or join tables."""

INDEX_MODEL_CONTRACT = 'Use logical_records as the source-family record lists; logical_table_index retains\nphysical lineage. Families come from actual source labels, not guessed file positions\nor examples. All returned records remain; source selection is not precomputed. Define a query-scoped master domain with eligibility flags;\nrelations are facts, not new entities. Resource amounts are already unit-prepared as described in unit_preparation; do not convert twice. Preserve signed sums,\nsnapshot ordering, unconditional category bounds, activation and fee/bonus semantics.\nReturn a compact symbolic formulation with this exact logical-to-physical Data Mapping.'

INDEX_CODE_CONTRACT = "Read CSVQA_DATA['logical_records'] as an ordinary family-to-record-list dictionary.\nEach list already includes all returned fragments; no custom data function is supplied. Do not substitute positions or example file IDs. Keep source data and variable\ncontainers distinct. Never use a Gurobi constraint/expression as a Python truth value or\nput a decision variable into a parameter dictionary. Required missing source values\nare errors, not zero defaults. Query and source data take precedence over examples."

INDEX_BINDING = "CSVQA_DATA['logical_records'] is a plain dictionary: exact source-family name ->\ncomplete list of record envelopes, already grouped across all returned physical fragments.\nThere is no custom records API. Use ordinary dictionary/list iteration:\nfor family, records in CSVQA_DATA['logical_records'].items():\n    for record in records:\n        values = record['values']\n# For one actual family: CSVQA_DATA['logical_records'][family].\n# record['table_id'], record['source_row'], record['source_family'] and\n# record['source_position'] preserve lineage. They are not entity identifiers.\n# source_position recovers original returned-view order, not pre-extraction CSV order.\n# Use physical file_index/source_row lineage for CSV-order rules. Keep every fragment.\n\nThe unit_preparation notice describes a prepared numeric view: usage and capacity\namount/unit fields now share the displayed unit per resource, using only literal Query\nequivalences. Use these numeric amounts directly; do not apply unit conversion again.\nAll source rows/event fields remain. Query scope/cutoff, highest revision, DELETE and\ndeduplication still need implementing in the Query's exact order before parameter parsing.\nRead from the resulting effective records, not a second raw-source iteration.\nFor exact original-value duplicates, source_values_sha256 preserves the preconversion\nidentity; it must not be replaced by a hash of normalized amounts. For original-unit\nfilters, overlay source_unit_fields on values to recover original values when present.\n\nCreate scalar quantity/selection variables on the query-scoped master IDs, including\nineligible IDs but fixing their quantity AND selection to zero according to the Query.\nIf a relation endpoint is outside that domain, use these patterns (not raw dict lookup):\n# incompatibility: constrain only when both endpoints have selection variables.\n# prerequisite: absent dependent means no constraint; absent prerequisite fixes\n# the existing dependent's selection to zero, otherwise y[dependent] <= y[prerequisite].\n# bonus: create variable and include objective term ONLY for a pair with both endpoints.\nbonus_vars = {}\nfor pair, coefficient in bonus_coefficients.items():\n    left, right = pair\n    if left not in y or right not in y:\n        continue\n    w = model.addVar(vtype=GRB.BINARY)\n    model.addConstr(w <= y[left])\n    model.addConstr(w <= y[right])\n    model.addConstr(w >= y[left] + y[right] - 1)\n    bonus_vars[pair] = w\n# Sum bonus_coefficients[pair] * bonus_vars[pair] ONLY over bonus_vars.\n# Coefficients are values, never part of the pair key. Category quantity bounds stated\n# unconditionally in the Query stay unconditional; activation links/fees are separate.\n# Implement zero-or-lot bounds with linear inequalities, never Python min/max on variables:\nfor i in I:\n    model.addConstr(q[i] >= minimum_lot[i] * y[i])\n    model.addConstr(q[i] <= maximum_order[i] * y[i])\n    model.addConstr(y[i] <= eligible[i])\n# eligible[i] is the Query's Boolean eligibility test, not bool(raw_string).\n# If category activation means any member selected, binary z[c] is OR:\n# y[i] <= z[c] for every member i; z[c] <= sum(y[i] for members i).\n# Never equate z[c] to that sum. Charge category_fee[c] * z[c] once per category.\nThese are binding/API patterns, not extra business constraints. Follow the actual Query.\nDo not read CSVQA_DATA['plan']; do not invent aliases, attributes or missing coefficients."
def _ask_extraction_plan(llm, route, query, tables, route_prompt):
    """Ask for one declarative plan and return the parsed plan plus raw response."""
    profile_data = _build_profile(tables, query)
    relational = _relational_export_layout(tables)
    if relational:
        for item, table in zip(profile_data, tables):
            item["_source_field_domains"] = _export_field_domains(table)
    profile = json.dumps(profile_data, ensure_ascii=False)
    request = f"""You are a CSV extraction planner. Return one JSON object only.
Choose tables, columns, and filters from the user query and the supplied CSV profile. Do not return
data, Python, formulas, prose, joins, aggregation, sorting, sampling, or invented fields.

Shared plan schema:
{{"route":"{route}","tables":[{{"file_index":0,"role":"descriptive non-unique role",
"columns":["exact column"] or "*","filters":{{"logic":"and|or","conditions":[
{{"column":"exact column","operator":"exact|prefix|contains|in|not_in|eq|ne|gt|ge|lt|le|between|is_null|not_null",
"dtype":"string|number|date","value":"scalar or list","evidence":"exact query span",
"format":"optional date format","inclusive":"both|left|right|neither"}}]}}}}],
"ignored_file_indices":[],"relationships":[{{"type":"matrix",
"matrix_table_id":"file_2_view_0","row_id_column":"exact column",
"row_axis":{{"table_id":"file_0_view_0","id_column":"exact column"}},
"column_axis":{{"table_id":"file_1_view_0","id_column":"exact column"}}}}]}}

Python assigns each logical view a table_id of file_<file_index>_view_<zero-based occurrence for that
file>. A physical file may appear in multiple table specifications and roles may repeat. Every input
file must appear in tables or ignored_file_indices. Ignore a file only when the query does not need it.
Filter only an explicit restriction; names introduced as examples (e.g., such as, for example)
do not restrict the entity set. Every filter must quote the query restriction that requires it.
Choose the filter operator from the requested matching semantics. A categorical code
classifying identifier families refers to a prefix when no separate category column exists;
arbitrary substring matching also admits unrelated identifiers and requires explicit query support.
Declare a matrix relationship only when the profile exposes its two axes unambiguously. When one label
column contains both decision entities and scalar parameter rows, retain every row that supplies a
query-required coefficient, price, cost, capacity, or bound; use separate logical views when helpful.

Route guidance:
{route_prompt}

User query:
{query}

CSV profile:
{profile}
"""
    if relational:
        request = INDEX_PLANNER_CONTRACT + "\n\n" + request
    text = str(llm.invoke([HumanMessage(content=request)]).content or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I) if text.startswith("```") else text
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start < 0 or end < start:
        error = ValueError(f"CSVQA planner did not return a JSON object: {text[:200]}")
        error.planner_output = text
        raise error
    try:
        plan = json.loads(cleaned[start:end + 1])
    except ValueError as exc:
        error = ValueError(f"CSVQA planner returned invalid JSON: {exc}")
        error.planner_output = text
        raise error from exc
    if not isinstance(plan, dict):
        raise ValueError("CSVQA extraction plan must be a JSON object")
    return plan, text


def _apply_condition(series, condition):
    """Apply one allowlisted string, number, date, range, set, or null predicate."""
    operator, dtype = condition["operator"], condition.get("dtype", "string")
    raw, missing = series.astype(str).str.strip(), series.astype(str).str.strip().eq("")
    if operator in {"is_null", "not_null"}:
        return missing if operator == "is_null" else ~missing
    if dtype == "number":
        values, convert = pd.to_numeric(raw, errors="coerce"), float
    elif dtype == "date":
        date_format = condition.get("format") or None
        values = pd.to_datetime(raw, format=date_format, errors="coerce")
        convert = lambda value: pd.to_datetime(value, format=date_format)
    else:
        values = raw.str.casefold()
        convert = lambda value: str(value).strip().casefold()
    if dtype != "string" and values.loc[~missing].isna().any():
        raise ValueError(f"Cannot reliably parse {series.name!r} as {dtype}")
    value = condition.get("value")
    wanted = [convert(item) for item in value] if isinstance(value, list) else convert(value)
    if operator in {"in", "not_in", "between"} and not isinstance(wanted, list):
        raise ValueError(f"Operator {operator!r} requires a list value")
    if operator == "between" and len(wanted) != 2:
        raise ValueError("Operator 'between' requires exactly two bounds")
    operations = {
        "exact": lambda: values.eq(wanted), "eq": lambda: values.eq(wanted),
        "ne": lambda: values.ne(wanted) & ~missing,
        "prefix": lambda: values.str.startswith(wanted, na=False),
        "contains": lambda: values.str.contains(wanted, regex=False, na=False),
        "in": lambda: values.isin(wanted), "not_in": lambda: ~values.isin(wanted) & ~missing,
        "gt": lambda: values.gt(wanted), "ge": lambda: values.ge(wanted),
        "lt": lambda: values.lt(wanted), "le": lambda: values.le(wanted),
        "between": lambda: values.between(
            wanted[0], wanted[1], inclusive=condition.get("inclusive", "both")
        ),
    }
    if operator not in operations:
        raise ValueError(f"Unsupported non-null filter operator: {operator}")
    return operations[operator]()


def _execute_plan(plan, tables, query, route):
    """Validate and execute a query-driven plan; validate only declared relationships."""
    key = lambda value: "".join(ch for ch in str(value or "").casefold() if ch.isalnum())
    route = normalize_route(route)
    if normalize_route(plan.get("route")) != route:
        raise ValueError(f"CSVQA plan route does not match {route}")
    specs = plan.get("tables")
    if not isinstance(specs, list) or not specs or any(not isinstance(spec, dict) for spec in specs):
        raise ValueError("CSVQA plan requires a non-empty tables list")
    ignored = plan.get("ignored_file_indices") or []
    if not isinstance(ignored, list) or any(isinstance(i, bool) or not isinstance(i, int) for i in ignored):
        raise ValueError("ignored_file_indices must be a list of integers")
    by_index = {table["file_index"]: table for table in tables}
    used = {spec.get("file_index") for spec in specs}
    if used & set(ignored) or (used | set(ignored)) != set(by_index):
        raise ValueError("Every input file must be used or explicitly ignored exactly once")

    payload_tables, payload_by_id, used_conditions, view_counts = [], {}, [], {}
    for spec in specs:
        file_index, role = spec.get("file_index"), str(spec.get("role") or "data").strip()
        if isinstance(file_index, bool) or not isinstance(file_index, int) or file_index not in by_index:
            raise ValueError(f"Invalid CSV file_index: {file_index!r}")
        occurrence = view_counts.get(file_index, 0)
        view_counts[file_index] = occurrence + 1
        table_id = f"file_{file_index}_view_{occurrence}"
        table, frame = by_index[file_index], by_index[file_index]["frame"]
        requested = spec.get("columns", "*")
        columns = list(frame.columns) if requested == "*" or requested == ["*"] else requested
        if not isinstance(columns, list) or not columns or len(columns) != len(set(columns)):
            raise ValueError(f"Table {table_id!r} requires unique columns or '*'")
        unknown = [column for column in columns if column not in frame.columns]
        if unknown:
            raise ValueError(f"Unknown columns for table {table_id!r}: {unknown}")
        filters = spec.get("filters") or {}
        if not isinstance(filters, dict):
            raise ValueError(f"Filters for table {table_id!r} must be an object")
        logic, conditions = str(filters.get("logic", "and")).casefold(), filters.get("conditions") or []
        if not isinstance(conditions, list):
            raise ValueError(f"Invalid filter logic for table {table_id!r}")
        # A common schema slip puts the sole condition's operator in ``logic``.
        # It is unambiguous for one condition; multi-condition plans remain strict.
        if logic not in {"and", "or"}:
            sole_operator = str(conditions[0].get("operator", "")).casefold() if len(conditions) == 1 and isinstance(conditions[0], dict) else ""
            if logic != sole_operator:
                raise ValueError(f"Invalid filter logic for table {table_id!r}")
            logic = "and"
            filters = {**filters, "logic": logic}
        combined = None
        for condition in conditions:
            if not isinstance(condition, dict):
                raise ValueError(f"Invalid filter for table {table_id!r}: {condition!r}")
            column, operator = condition.get("column"), condition.get("operator")
            dtype, evidence = condition.get("dtype", "string"), str(condition.get("evidence") or "")
            if column not in frame.columns or operator not in FILTER_OPERATORS or dtype not in FILTER_DTYPES:
                raise ValueError(f"Invalid filter for table {table_id!r}: {condition}")
            raw_values = condition.get("value", [])
            raw_values = raw_values if isinstance(raw_values, list) else [raw_values]
            value_supported = bool(raw_values) and all(key(value) in key(query) for value in raw_values)
            if not evidence.strip() or (key(evidence) not in key(query) and not value_supported):
                raise ValueError(f"Filter lacks query evidence: {condition}")
            illustrative = re.search(r"\bsuch as\b|\bfor example\b|\be\.g\.", evidence, re.I)
            restrictive = re.search(r"\bonly\b|\bexclud\w*\b|\bexcept\b|\brestrict\w*\b|\blimited to\b", evidence, re.I)
            if illustrative and not restrictive:
                raise ValueError(f"Illustrative query examples cannot justify a source-row filter: {evidence}")
            mask = _apply_condition(frame[column], condition)
            combined = mask if combined is None else (combined & mask if logic == "and" else combined | mask)
            used_conditions.append(condition)
        # Preserve source identity needed by filters and declared matrix axes.
        columns = list(columns)
        relationships = plan.get("relationships") or []
        matrix_specs = [r for r in relationships if isinstance(r, dict) and r.get("matrix_table_id") == table_id]
        identity_columns = [] if matrix_specs else [c.get("column") for c in conditions]
        for relationship in relationships:
            if not isinstance(relationship, dict):
                continue
            if relationship.get("matrix_table_id") == table_id:
                identity_columns.append(relationship.get("row_id_column"))
            for axis_name in ["row_axis", "column_axis"]:
                axis = relationship.get(axis_name) or {}
                if isinstance(axis, dict) and axis.get("table_id") == table_id:
                    identity_columns.append(axis.get("id_column"))
        for column in identity_columns:
            if column in frame.columns and column not in columns:
                columns.append(column)
        selected = frame if combined is None else frame.loc[combined]
        if selected.empty:
            raise ValueError(f"CSVQA plan returned zero rows for table {table_id!r}")
        records = [
            {"source_row": int(index), "values": {column: str(row[column]) for column in columns}}
            for index, row in selected.iterrows()
        ]
        payload = {
            "table_id": table_id, "file_index": file_index, "file_name": table["path"].name,
            "role": role or "data", "columns": columns, "original_rows": len(frame),
            "returned_rows": len(records), "filters": filters, "records": records,
        }
        payload_tables.append(payload)
        payload_by_id[table_id] = payload

    profile = _build_profile(tables, query)
    required_columns = {
        column for file_profile in profile for matches in file_profile["query_terms"].values()
        for column in matches["column_matches"]
    }
    selected_columns = {column for table in payload_tables for column in table["columns"]}
    missing_columns = sorted(required_columns - selected_columns)
    if missing_columns:
        raise ValueError(f"Plan omitted columns named by the query: {missing_columns}")

    relationships = plan.get("relationships") or []
    if not isinstance(relationships, list):
        raise ValueError("relationships must be a list")
    validated_relationships, matrix_checks = [], []
    for relationship in relationships:
        if not isinstance(relationship, dict) or relationship.get("type") != "matrix":
            raise ValueError(f"Unsupported relationship: {relationship!r}")
        matrix_id = relationship.get("matrix_table_id")
        row_axis_spec, column_axis_spec = relationship.get("row_axis") or {}, relationship.get("column_axis") or {}
        if not isinstance(row_axis_spec, dict) or not isinstance(column_axis_spec, dict):
            raise ValueError("Matrix row_axis and column_axis must be objects")
        row_axis_id, column_axis_id = row_axis_spec.get("table_id"), column_axis_spec.get("table_id")
        if any(table_id not in payload_by_id for table_id in (matrix_id, row_axis_id, column_axis_id)):
            raise ValueError(f"Relationship references an unknown table_id: {relationship}")
        matrix_table, row_axis, column_axis = (
            payload_by_id[matrix_id], payload_by_id[row_axis_id], payload_by_id[column_axis_id]
        )
        row_id_column = relationship.get("row_id_column")
        row_axis_column, column_axis_column = row_axis_spec.get("id_column"), column_axis_spec.get("id_column")
        for table, column in (
            (matrix_table, row_id_column), (row_axis, row_axis_column), (column_axis, column_axis_column)
        ):
            if column not in table["columns"]:
                raise ValueError(f"Matrix ID column {column!r} is not selected for {table['table_id']!r}")

        def axis_keys(values, axis_name):
            normalized = [str(value) for value in values]
            if any(not value for value in normalized) or len(set(normalized)) != len(normalized):
                raise ValueError(f"Matrix {axis_name} IDs must be non-empty and unique: {values}")
            return normalized

        row_ids = [record["values"][row_axis_column] for record in row_axis["records"]]
        column_ids = [record["values"][column_axis_column] for record in column_axis["records"]]
        matrix_row_ids = [record["values"][row_id_column] for record in matrix_table["records"]]
        matrix_columns = [column for column in matrix_table["columns"] if column != row_id_column]
        def axis_mapping(actual, expected):
            axis_keys(actual, "source")
            axis_keys(expected, "reference")
            if set(actual) == set(expected):
                return dict(zip(actual, actual)), "exact"
            def suffix_map(values):
                matches = [re.fullmatch(r"\D*(\d+)", value) for value in values]
                if not all(matches):
                    return None
                keys = [match.group(1) for match in matches]
                return dict(zip(keys, values)) if len(set(keys)) == len(keys) else None
            left, right = suffix_map(actual), suffix_map(expected)
            if left is not None and right is not None and set(left) == set(right):
                return {value: right[key] for key, value in left.items()}, "unique_complete_suffix"
            return None, "unresolved"
        row_map, row_basis = axis_mapping(matrix_row_ids, row_ids)
        column_map, column_basis = axis_mapping(matrix_columns, column_ids)
        relationship = dict(relationship)
        if row_basis == "unique_complete_suffix":
            relationship["row_id_mapping"] = row_map
        if column_basis == "unique_complete_suffix":
            relationship["column_id_mapping"] = column_map
        checks = {
            "matrix_table_id": matrix_id,
            "shape": [len(matrix_table["records"]), len(matrix_columns)],
            "expected_shape": [len(row_ids), len(column_ids)],
            "row_ids_aligned": row_map is not None, "column_ids_aligned": column_map is not None,
            "row_mapping_basis": row_basis, "column_mapping_basis": column_basis,
        }
        if checks["shape"] != checks["expected_shape"] or not checks["row_ids_aligned"] or not checks["column_ids_aligned"]:
            raise ValueError(f"Matrix validation failed: {checks}")
        validated_relationships.append(relationship)
        matrix_checks.append(checks)

    payload_tables.sort(key=lambda table: (table["file_index"], table["table_id"]))
    validation = {"status": "OK", "matrix_checks": matrix_checks}
    return {
        "route": route, "query": str(query or ""), "tables": payload_tables,
        "relationships": validated_relationships, "ignored_file_indices": sorted(ignored),
        "validation": validation,
    }

"""Offline prototype: extract literal unit equalities with exact Query provenance.

No table values, scope, parameters, programs or solver models are changed here.
Unknown prose is left unsupported, not guessed. This is not yet an active candidate.
"""
import math
import re
_UV_NUMBER = '(?:\\d+(?:\\.\\d*)?|\\.\\d+)(?:[eE][+-]?\\d+)?'
_UV_WORD = '[A-Za-z][A-Za-z0-9_/-]*'
_UV_ENGLISH_UNIT_WORDS = frozenset('second minute hour day week month year millisecond microsecond liter litre milliliter millilitre meter metre kilometer kilometre centimeter centimetre millimeter millimetre gram kilogram tonne ton watt kilowatt joule ampere volt byte bit'.split())
_UV_PER = re.compile('(?<![\\w.,+-])(' + _UV_NUMBER + ')\\s+(' + _UV_WORD + ')\\s+per\\s+(' + _UV_WORD + ')(?!\\w)', re.I)
_UV_EQUALITY = re.compile('(?<![\\w.,+-])(' + _UV_NUMBER + ')\\s*(' + _UV_WORD + ')\\s*=\\s*(' + _UV_NUMBER + ')\\s*(' + _UV_WORD + ')(?!\\w)', re.I)

def _uview_label(token, observed_units):
    """Resolve only observed spellings, case, or one unambiguous English plural."""
    if token in observed_units:
        return token
    word = token.casefold()
    exact_casefold = [u for u in observed_units if word in _UV_ENGLISH_UNIT_WORDS and u.casefold() == word]
    if len(exact_casefold) == 1:
        return exact_casefold[0]
    if len(exact_casefold) > 1:
        raise ValueError('Ambiguous observed unit spelling: ' + token)
    singular = token[:-1] if word.endswith('s') and word[:-1] in _UV_ENGLISH_UNIT_WORDS else None
    plural = [u for u in observed_units if singular and u.casefold() == singular.casefold()]
    if len(plural) == 1:
        return plural[0]
    if len(plural) > 1:
        raise ValueError('Ambiguous observed plural unit: ' + token)
    return None

def _uview_conversions(query, observed_units):
    """Return only explicitly written ratios between known source-unit labels.

    Supported forms: ``1000 ml per liter`` or ``1 liter = 1000 ml``.
    Every output retains the exact Query span and its offsets; no unit list/ratio
    is benchmark-specific, and no conversion involving a resource name is inferred.
    """
    query = str(query)
    observed = tuple(dict.fromkeys((str(u) for u in observed_units if str(u))))
    raw = []
    for match in _UV_PER.finditer(query):
        factor, target, source = match.groups()
        raw.append((match, source, target, float(factor), 'number target per source', factor, '1'))
    for match in _UV_EQUALITY.finditer(query):
        left_value, source, right_value, target = match.groups()
        denominator = float(left_value)
        if denominator <= 0 or not math.isfinite(denominator):
            raise ValueError('Literal conversion source quantity must be positive')
        raw.append((match, source, target, float(right_value) / denominator, 'literal equality', right_value, left_value))
    result, seen = ([], {})
    boundaries = list(re.finditer('(?<!\\d)[.!?;](?:\\s|$)|\\n', query))
    for match, source_token, target_token, factor, basis, numerator, denominator in sorted(raw, key=lambda x: x[0].start()):
        left = max([m.end() for m in boundaries if m.end() <= match.start()] or [0])
        right = min([m.start() for m in boundaries if m.start() >= match.end()] or [len(query)])
        statement = query[left:right]
        first_ratio = min((m.start() for m, *_ in raw if left <= m.start() and m.end() <= right))
        introduction = query[left:first_ratio].strip()
        conversion_intro = re.fullmatch('Convert\\s+(' + _UV_WORD + ')\\s+to\\s+(' + _UV_WORD + ')\\s+using', introduction, re.I)
        affirmative = introduction.casefold() in ('', 'use')
        if conversion_intro:
            intro_units = {_uview_label(token, observed) for token in conversion_intro.groups()}
            actual_units = {_uview_label(source_token, observed), _uview_label(target_token, observed)}
            affirmative = None not in intro_units and intro_units == actual_units
        if not affirmative or re.search('\\b(?:not|no|never|ignore|ignored|avoid|obsolete|previous|previously|formerly|old|example|examples|quoted|superseded)\\b|[\\"“”]', statement, re.I):
            continue
        source = _uview_label(source_token, observed)
        target = _uview_label(target_token, observed)
        if source is None or target is None:
            continue
        if not math.isfinite(factor) or factor <= 0:
            raise ValueError('Literal conversion factor must be positive and finite')
        pair = (source, target)
        if source == target and (not math.isclose(factor, 1, rel_tol=1e-12, abs_tol=1e-12)):
            raise ValueError('Literal identity conversion must have factor one')
        if pair in seen and (not math.isclose(seen[pair], factor, rel_tol=1e-12, abs_tol=1e-12)):
            raise ValueError('Conflicting literal Query ratios')
        if (target, source) in seen and (not math.isclose(seen[target, source] * factor, 1, rel_tol=1e-12, abs_tol=1e-12)):
            raise ValueError('Conflicting reciprocal literal Query ratios')
        seen[pair] = factor
        result.append({'source_unit': source, 'target_unit': target, 'factor': factor, 'ratio_numerator': numerator, 'ratio_denominator': denominator, 'evidence': match.group(0), 'query_span': [match.start(), match.end()], 'basis': basis})
    return result
'Explicit, provenance-preserving unit preparation; no scope/snapshot/model logic.'
import copy
from decimal import Decimal, DecimalException
from fractions import Fraction
import hashlib
import json

def _uview_prepare(payload, query):
    """Return a new equivalent amount/unit view and a complete transformation report.

    Source objects are not mutated. Only rows with nonempty resource, amount and
    unit fields can be transformed. The Query's literal equivalences are the only
    graph edges. Unknown multi-unit resource groups fail explicitly; missing
    numeric fields are never filled. Event revisions/actions/keys are untouched.
    """
    if 'unit_preparation' in payload:
        raise ValueError('Unit view is already prepared; retain its original provenance')
    units = list(dict.fromkeys((str(r['values']['unit']) for t in payload['tables'] for r in t['records'] if r['values'].get('unit'))))
    evidence = _uview_conversions(query, units)
    graph = {}
    for position, item in enumerate(evidence):
        factor = Fraction(Decimal(item['ratio_numerator'])) / Fraction(Decimal(item['ratio_denominator']))
        a, b = (item['source_unit'], item['target_unit'])
        graph.setdefault(a, []).append((b, factor, position))
        graph.setdefault(b, []).append((a, 1 / factor, position))
    targets, paths = ({}, {})
    for origin in units:
        factors, routes, pending = ({origin: Fraction(1)}, {origin: []}, [origin])
        while pending:
            current = pending.pop()
            for neighbor, factor, position in graph.get(current, []):
                candidate = factors[current] * factor
                if neighbor in factors:
                    if factors[neighbor] != candidate:
                        raise ValueError('Conflicting literal Query unit equivalences')
                else:
                    factors[neighbor] = candidate
                    routes[neighbor] = routes[current] + [position]
                    pending.append(neighbor)
        target = min(factors, key=lambda unit: (-factors[unit], unit))
        targets[origin] = (target, factors[target])
        paths[origin] = routes[target]
    prepared = copy.deepcopy(payload)
    changes, resource_units = ([], {})
    for table in prepared['tables']:
        for record in table['records']:
            values = record['values']
            record['source_values_sha256'] = hashlib.sha256(json.dumps(values, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
            if values.get('amount') not in ('', None) and bool(values.get('resource')) != bool(values.get('unit')):
                raise ValueError('A resource amount must retain both resource and unit source fields')
            if not all((values.get(key) not in ('', None) for key in ('resource', 'amount', 'unit'))):
                continue
            resource, unit, raw_amount = (str(values['resource']), str(values['unit']), str(values['amount']))
            try:
                amount = Decimal(raw_amount)
            except DecimalException as exc:
                raise ValueError('Resource amount is not a valid source number: ' + raw_amount) from exc
            if not amount.is_finite():
                raise ValueError('Nonfinite resource amount in source record')
            target, factor = targets[unit]
            resource_units.setdefault(resource, set()).add(target)
            if factor == 1 and unit == target:
                continue
            converted = Fraction(amount) * factor
            denominator, twos, fives = (converted.denominator, 0, 0)
            while denominator % 2 == 0:
                denominator //= 2
                twos += 1
            while denominator % 5 == 0:
                denominator //= 5
                fives += 1
            if denominator != 1:
                raise ValueError('Prepared unit view requires an exact finite decimal; no rounding is allowed')
            scale = max(twos, fives)
            integer = converted.numerator * 2 ** (scale - twos) * 5 ** (scale - fives)
            digits = str(abs(integer)).zfill(scale + 1)
            converted_text = ('-' if integer < 0 else '') + (digits[:-scale] + '.' + digits[-scale:] if scale else digits)
            values['amount'], values['unit'] = (converted_text, target)
            record['source_unit_fields'] = {'amount': raw_amount, 'unit': unit}
            changes.append({'table_id': table['table_id'], 'source_row': record['source_row'], 'original_amount': raw_amount, 'original_unit': unit, 'prepared_amount': converted_text, 'prepared_unit': target, 'factor_numerator': str(factor.numerator), 'factor_denominator': str(factor.denominator), 'evidence_indices': paths[unit]})
    unresolved = {r: sorted(u) for r, u in resource_units.items() if len(u) > 1}
    if unresolved:
        raise ValueError('Query has no complete explicit unit path for resource groups: ' + json.dumps(unresolved))
    raw_sha = hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    summary = {'version': 'literal-query-unit-view-1', 'source_observation_sha256': raw_sha, 'scope_snapshot_dedup_applied': False, 'converted_rows': len(changes), 'literal_equivalences': evidence, 'resource_units': {r: next(iter(u)) for r, u in sorted(resource_units.items())}, 'instruction': 'This is an explicit prepared unit view. Resource amount/unit fields are already converted using only the literal Query equivalences. Usage and ledger for each resource now share the displayed unit; use these amounts as-is, without converting twice. All original rows, identifiers, event fields and selection rules remain; no scope, snapshot, DELETE, deduplication or eligibility operation was applied. For an exact original-value duplicate rule use source_values_sha256, which predates preparation; query-defined composite dedup keys still take precedence. Original amount/unit are also in source_unit_fields when changed; reconstruct original values by overlaying them on values if an original-field filter requires it. The complete original observation and per-row conversions are preserved in the trace.'}
    prepared['unit_preparation'] = summary
    report = {**summary, 'transformations': changes}
    return (prepared, report)

def _validate_named_source_coverage(payload, source_tables, query):
    """Reject omitted query-named source families; native full-data fallback remains authoritative."""
    if not _relational_export_layout(source_tables):
        return
    used = {t["file_index"] for t in payload["tables"]}
    missing = []
    for table in source_tables:
        if "table" not in table["frame"].columns:
            continue
        labels = list(dict.fromkeys(str(v) for v in table["frame"]["table"] if str(v)))
        named = [label for label in labels if re.search(
            r"(?<!\w)" + re.escape(label) + r"(?!\w)", str(query), re.I)]
        if named and table["file_index"] not in used:
            missing.append({"file_index": table["file_index"], "query_named_labels": named})
    if missing:
        raise ValueError("Extraction omitted query-named source fragments: " + json.dumps(missing))




"""A plain-data family view; no API, selection, aggregation, or solver logic."""
import copy

def _logical_records_view(payload):
    """Group each returned record once, preserving values and reversible lineage."""
    if 'logical_table_index' not in payload:
        return payload
    if 'logical_records' in payload:
        raise ValueError('Logical records are already prepared')
    groups = {family: [] for family in payload['logical_table_index']}
    position, seen = (0, set())
    for table in payload['tables']:
        table_id = table['table_id']
        evidence = payload['logical_table_evidence'][table_id]
        for record in table['records']:
            identity = (table_id, record['source_row'])
            if identity in seen:
                raise ValueError('Duplicate table_id/source_row in returned observation')
            seen.add(identity)
            if any((k in record for k in ('table_id', 'source_family', 'source_position'))):
                raise ValueError('Reserved derived-record lineage key already exists')
            family = record['values'].get('table')
            if family is None:
                labels = evidence['source_labels']
                family = labels[0] if len(labels) == 1 else evidence['record_labels'][str(record['source_row'])]
            if family not in groups or table_id not in payload['logical_table_index'][family]:
                raise ValueError('Returned record has no exact source-family lineage')
            groups[family].append({**copy.deepcopy(record), 'table_id': table_id, 'source_family': family, 'source_position': position})
            position += 1
    result = dict(payload)
    result['logical_records'] = groups
    result['logical_records_metadata'] = {'version': 'plain-family-records-1', 'record_count': position, 'family_record_counts': {k: len(v) for k, v in groups.items()}, 'rows_selected_aggregated_or_deduplicated': False, 'instruction': 'logical_records is an ordinary dictionary of complete record lists, not a function. Each returned table_id/source_row occurs once. Record fields are in values; source_family/table_id/source_position are lineage, not decision entities. Sorting all records by source_position recovers the original returned source order. Scope, snapshot, DELETE, deduplication, aggregation and optimization are not performed here.'}
    return result


## 5. CSVQA tool

Planned mode supplies structured data and falls back to the full data if the extraction plan is invalid. Legacy mode sends every CSV row in source order directly to the evidence model.


In [ ]:
def load_csv_documents(dataset_address):
    """Build one legacy document per source row in deterministic source order."""
    documents, source_order = [], 0
    for table in _load_tables(dataset_address):
        for source_row, row in table["frame"].iterrows():
            values = {str(column): str(row[column]) for column in row.index}
            documents.append(Document(
                page_content=json.dumps({"source_row": int(source_row), "values": values}, ensure_ascii=False),
                metadata={"source": str(table["path"]), "source_order": source_order},
            ))
            source_order += 1
    return documents


def build_csvqa_components(
    dataset_address, system_prompt, tool_description, *, route, user_query="",
    result_sink=None,
):
    """Build the shared CSVQA tool in query-planned or legacy full-data mode."""
    route = normalize_route(route)
    planned = CSVQA_MODE_BY_ROUTE[route] == "planned"
    llm = make_llm()

    state = result_sink if result_sink is not None else {}

    def rag_answer(tool_query):
        documents = load_csv_documents(dataset_address)
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            (
                "human",
                "Original user query (the only authority for subsets):\n{user_query}\n\n"
                "Agent retrieval request (must not narrow beyond the original query):\n{input}",
            ),
        ])
        result = create_stuff_documents_chain(llm, prompt).invoke({
            "input": tool_query, "user_query": user_query, "context": documents,
        })
        if not str(result or "").strip():
            raise RuntimeError("Legacy CSVQA model returned an empty result")
        return str(result)

    def qa_wrapper(tool_query):
        plan, planner_outputs, errors, status, fallback_reason = None, [], [], "LEGACY_FULL_DATA", None
        attempt_count = 0
        if planned:
            tables = _load_tables(dataset_address)
            attempt_count = 1
            raw_output = ""
            try:
                plan, raw_output = _ask_extraction_plan(
                    llm, route, user_query, tables, system_prompt
                )
                payload = _execute_plan(plan, tables, user_query, route)
                _validate_named_source_coverage(payload, tables, user_query)
                planner_outputs.append(raw_output)
                observation = json.dumps(payload, ensure_ascii=False, sort_keys=True)
                status = "PLANNED"
            except (KeyError, TypeError, ValueError) as exc:
                planner_outputs.append(getattr(exc, "planner_output", raw_output))
                errors.append(str(exc))
                if CSVQA_MODE_BY_ROUTE[route] != "planned":
                    raise ValueError(
                        f"CSVQA extraction plan failed for {route}: {errors[-1]}"
                    ) from exc
                payload_tables = []
                for table in tables:
                    frame, columns = table["frame"], list(table["frame"].columns)
                    payload_tables.append({
                        "table_id": f"file_{table['file_index']}_view_0",
                        "file_index": table["file_index"], "file_name": table["path"].name,
                        "role": f"file_{table['file_index']}", "columns": columns,
                        "original_rows": len(frame), "returned_rows": len(frame),
                        "filters": {"logic": "and", "conditions": []},
                        "records": [{"source_row": int(index), "values": {
                            column: str(row[column]) for column in columns
                        }} for index, row in frame.iterrows()],
                    })
                fallback_reason = errors[-1] if errors else "planner did not produce a valid plan"
                observation = json.dumps({
                    "route": route, "query": str(user_query or ""), "tables": payload_tables,
                    "relationships": [], "ignored_file_indices": [],
                    "validation": {
                        "status": "FALLBACK_FULL_DATA", "planner_errors": errors,
                        "fallback_reason": fallback_reason,
                    },
                }, ensure_ascii=False, sort_keys=True)
                status = "FALLBACK_FULL_DATA"
        else:
            observation = rag_answer(tool_query)
        if planned:
            payload = _annotate_logical_tables(json.loads(observation), tables)
            observation = json.dumps(payload, ensure_ascii=False, sort_keys=True)
        trace = {
            "version": "GPT4.1-LO", "mode": "planned" if planned else "legacy",
            "status": status, "plan": plan, "planner_attempt_count": attempt_count,
            "planner_outputs": planner_outputs, "planner_errors": errors,
            "validation_errors": errors, "fallback_reason": fallback_reason,
            "repair_count": 0, "retry_count": 0,
            "fallback_count": int(status == "FALLBACK_FULL_DATA"),
            "payload_hash": hashlib.sha256(observation.encode("utf-8")).hexdigest(),
        }
        if planned and _relational_export_layout(tables):
            raw_observation = observation
            try:
                prepared, preparation = _uview_prepare(json.loads(raw_observation), user_query)
                prepared = _annotate_logical_tables(prepared, tables)
                prepared = _logical_records_view(prepared)
                observation = json.dumps(prepared, ensure_ascii=False, sort_keys=True)
            except (KeyError, TypeError, ValueError) as exc:
                trace.update(status="UNIT_PREPARATION_FAILED", raw_observation=raw_observation,
                             raw_payload_hash=trace["payload_hash"], unit_preparation_error=str(exc))
                state.clear()
                state.update({"observation": raw_observation, "trace": trace})
                raise
            trace.update(raw_observation=raw_observation, raw_payload_hash=trace["payload_hash"],
                         unit_preparation=preparation,
                         payload_hash=hashlib.sha256(observation.encode("utf-8")).hexdigest())
        state.clear()
        state.update({"observation": observation, "trace": trace})
        return observation

    return llm, Tool(name="CSVQA", func=qa_wrapper, description=tool_description), state


## 6. Problem classification

Retrieve reference examples and classify the query. The classifier and its retriever are cached with lru_cache to avoid rebuilding indexes for each case.


In [ ]:

@lru_cache(maxsize=1)
def _get_classifier_retriever():
    documents = CSVLoader(
        file_path=str(PROJECT_ROOT / RAG_EXAMPLES_ALL_PATH),
        encoding="utf-8-sig",
        content_columns=["prompt", "Type"],
    ).load()
    if len(documents) < 5:
        raise ValueError("Classification reference requires at least five labeled examples")
    return FAISS.from_documents(documents, make_embeddings()).as_retriever(
        search_kwargs={"k": 5}
    )


def fileqa_retrieve(query):
    """Retrieve five raw classification examples without an additional LLM call.
    
    Args:
        query: Natural-language optimization problem or retrieval query.
    
    Returns:
        A formatted string containing the ranked retrieved examples.
    """
    retrieved = _get_classifier_retriever().invoke(str(query))
    if len(retrieved) != 5:
        raise RuntimeError(f"FileQA expected five examples, retrieved {len(retrieved)}")

    return "\n\n".join(
        f"[Retrieved example {rank}]\n{document.page_content}"
        for rank, document in enumerate(retrieved, start=1)
    )


few_shot_example = """
Example 1 — Network Revenue Management

Question:
What is the problem type of the following optimization problem? Text: There
are three products with corresponding revenues and independent demand
streams. The company decides how many requests to fulfil over a finite sales
horizon to maximize total revenue from fixed initial inventories.
Replenishment is not allowed, and fulfilment decisions are non-negative
integers.

Thought: I need to identify the mathematical structure. I should call FileQA
once to retrieve structurally similar labeled reference problems.

Action: FileQA

Action Input: finite-horizon revenue maximization with independent demand streams, fixed initial inventories, no replenishment, and integer demand-fulfilment decisions

Observation: The retrieved RAG_Examples_All examples contain finite-inventory
demand-fulfilment problems labeled Network Revenue Management.

Thought: Fixed inventories are allocated to demand fulfilment over a sales
horizon to maximize revenue. This is the defining NRM structure.

Final Answer: NRM


Example 2 — Mixture rather than Resource Allocation

Question:
What is the problem type of the following optimization problem? Text: A
manufacturer determines integer production quantities and integer batch
counts on several workstations. Each batch consumes machine capacity.
Production creates a byproduct that must either be sold or disposed of, and
the corresponding quantities must satisfy balance constraints.

Thought: This problem contains resource capacities, but I must check whether
it is a clean single-stage resource-allocation model or contains additional
interacting mechanisms. I should call FileQA once.

Action: FileQA

Action Input: integer production and batch-count decisions across multiple workstations with capacity constraints and coupled byproduct sale-or-disposal balance

Observation: The retrieved examples include resource-allocation and
non-standard production-planning models. The latter contain batch decisions,
multiple production components, and coupled balance constraints.

Thought: Ordinary capacity constraints alone could indicate RA. However,
integer batch counts, multiple workstations, and byproduct balance materially
change the formulation. Therefore, the problem is Mixture rather than RA.

Final Answer: Mixture


Example 3 — Others rather than Mixture

Question:
What is the problem type of the following optimization problem? Text: A
company selects a subset of independent projects. Each project has a profit
and consumes part of one available budget. Each project is represented by a
binary variable, and there are no precedence, dependency, assignment, batch,
flow, or multi-stage constraints.

Thought: I need to determine whether this is a hybrid model or a coherent
standard optimization family outside the five canonical classes. I should
call FileQA once.

Action: FileQA

Action Input: independent binary project selection maximizing profit under one budget constraint with no logical dependencies or additional coupling

Observation: The retrieved examples contain standard binary knapsack and
project-selection problems categorized as outside the five canonical
classes.

Thought: This is a clean knapsack-style model without interacting structures
or non-standard coupling. It should be Others, not Mixture.

Final Answer: Others
"""


prefix = f"""
You are the classification agent in LEAN-LLM-OPT. Classify the current
optimization problem into exactly one of the following seven semantic
classes:

- NRM: Network Revenue Management
- RA: Resource Allocation
- TP: Transportation Problem
- FLP: Facility Location Problem
- AP: Assignment Problem
- Mixture
- Others

You must follow the ReAct workflow and call FileQA exactly once.

FileQA retrieves structurally similar labeled optimization problems from
RAG_Examples_All. For the Action Input, provide a concise but complete structural
description that preserves the objective, decision variables, major
constraints, and important coupling mechanisms.

Retrieved examples are supporting evidence, not automatic votes. The current
problem description and its mathematical structure are authoritative.

Classify according to:

- the objective;
- the meanings and domains of the decision variables;
- the principal constraint families;
- coupling between decisions;
- whether the formulation contains multiple stages or additional modeling
  mechanisms.

Do not classify according to company names, application narratives,
filenames, CSV files, or isolated keywords.

Class definitions:

1. Network Revenue Management (NRM)

A problem that allocates fixed inventories or capacities to known or
stochastic demand streams in order to maximize revenue. Typical defining
features include demand fulfilment, finite inventory, a sales horizon, and
no replenishment.

2. Resource Allocation (RA)

A clean, primarily single-stage allocation of limited resources among
activities or products, usually with ordinary capacity constraints and an
additive objective.

Capacity constraints or integer production quantities alone do not make a
problem Mixture.

Do not choose RA when additional mechanisms materially change the
formulation, such as:

- batch-size or lot-count decisions;
- multiple interacting production stages, workstations, or devices;
- byproduct sale, disposal, or material-balance decisions;
- blending or proportion constraints interacting with production;
- logical dependencies or conditional decisions;
- selection followed by assignment;
- equilibrium-style balance equations;
- vehicle-count and shipment-quantity coupling.

3. Transportation Problem (TP)

A clean source-to-destination flow problem with source supplies, destination
demands, route shipment quantities, and route transportation costs.

Do not choose TP when the transportation structure is materially coupled
with mechanisms such as integer truck dispatch, vehicle-capacity decisions,
fixed-charge decisions, multi-stage transshipment, production decisions, or
other non-standard constraint families. Such a problem may be Mixture.

4. Facility Location Problem (FLP)

A problem containing facility-opening or location decisions with fixed
opening costs, together with assignment or shipment decisions that link
clients or demand points to opened facilities.

5. Assignment Problem (AP)

A clean one-to-one or matching-style assignment of agents, workers,
machines, products, or other entities to tasks.

If the problem first selects a subset of entities and then assigns them, or
combines assignment with multiple stages, logical dependencies, production,
flow, or other interacting constraint families, choose Mixture instead.

6. Mixture

A hybrid or non-standard optimization model in which multiple structures or
modeling mechanisms interact materially.

Choose Mixture when either:

(a) a canonical TP, NRM, RA, FLP, or AP structure is combined with an
important additional mechanism; or

(b) a non-core optimization model contains non-standard interacting
constraint families that prevent it from being a clean textbook instance.

Strong Mixture signals include:

- batch-size, lot-size, or integer-trip decisions coupled with quantities;
- logical implications, precedence, dependencies, or conditional decisions;
- subset selection coupled with assignment;
- blending or proportion requirements coupled with production or capacity;
- byproduct production, sale, disposal, or balance;
- multiple interacting workstations, devices, or production stages;
- equilibrium or cross-variable balance equations;
- truck-count or vehicle-capacity decisions coupled with transportation flow;
- several distinct decision families whose interactions materially affect
  the formulation.

A problem does not need to belong to two named canonical classes to be
Mixture. One recognizable base structure plus a material non-standard
coupling mechanism is sufficient.

7. Others

A clean and coherent optimization family outside NRM, RA, TP, FLP, and AP,
without the hybrid or non-standard interacting structure required for
Mixture.

Typical examples include:

- a standard diet problem;
- a pure knapsack or independent project-selection problem;
- a standard traveling salesperson problem;
- a standard workforce or shift-covering problem;
- a standard parallel-machine scheduling problem;
- another well-known outside family with one dominant mathematical
  structure.

A textbook blending model may be Others only when it is a standalone model
with ordinary ingredient-balance and quality constraints. If blending or
proportion decisions interact with production stages, capacities, equipment,
batch decisions, or other constraint families, choose Mixture.

Decision procedure:

1. Identify the objective, decision variables, and main constraints.

2. Determine whether the problem is a clean instance of NRM, RA, TP, FLP,
   or AP.

3. Even if a canonical base structure is present, check for a material
   additional mechanism such as batching, logical dependencies,
   selection-assignment coupling, byproducts, blending-production coupling,
   multiple stages, equilibrium equations, or vehicle-count coupling.

4. If such an additional mechanism materially changes the formulation,
   choose Mixture.

5. If the problem is a clean, well-known outside family with one dominant
   structure and no material hybrid coupling, choose Others.

6. When TP, NRM, RA, FLP, or AP cannot be confidently confirmed:
   - choose Mixture if multiple structures or non-standard constraints
     interact materially;
   - otherwise choose Others.

When deciding between Mixture and Others, choose Mixture if two different decision families or constraint mechanisms interact materially. Choose Others only when one standard textbook archetype explains all important variables and constraints.

7. Retrieved labels are evidence, not rules. Do not use majority voting.
   When retrieval conflicts with the current mathematical formulation,
   prioritize the current formulation.

Important boundary rules:

- Multiple ordinary resource-capacity constraints alone are not sufficient
  for Mixture; a clean allocation model remains RA.
- Integer variables alone are not sufficient for Mixture.
- A canonical structure plus a material additional decision mechanism is
  sufficient for Mixture.
- Mixture and Others are separate semantic labels even if they currently
  share the same downstream workflow route.

Return exactly one of the following label tokens in the Final Answer:

NRM, RA, TP, FLP, AP, Mixture, Others

Do not include explanations, punctuation, or multiple candidate labels in
the Final Answer.

Examples of the required ReAct process:

{few_shot_example}
"""


suffix = """
Begin!

Question: {input}
{agent_scratchpad}
"""




def normalize_problem_class(value):
    """Normalize a raw label to one of the seven benchmark classes.
    
    Args:
        value: Input value to normalize, parse, or inspect.
    
    Returns:
        The canonical class label.
    """
    candidate = str(value or "").strip()
    final = re.findall(r"(?:Final Answer|Answer|Label|Class)\s*:\s*([^\n]+)", candidate, re.I)
    if final:
        candidate = final[-1].strip()
    upper = re.sub(r"\s+", " ", candidate.strip("`*_ .,:;").upper())
    aliases = {
        "TP": "TP", "TRANSPORTATION": "TP", "TRANSPORTATION PROBLEM": "TP",
        "NRM": "NRM", "NETWORK REVENUE MANAGEMENT": "NRM",
        "NETWORK REVENUE MANAGEMENT PROBLEM": "NRM",
        "RA": "RA", "RESOURCE ALLOCATION": "RA", "RESOURCE ALLOCATION PROBLEM": "RA",
        "FLP": "FLP", "UFLP": "FLP", "FACILITY LOCATION": "FLP",
        "FACILITY LOCATION PROBLEM": "FLP",
        "UNCAPACITATED FACILITY LOCATION PROBLEM": "FLP",
        "UNCAPACITED FACILITY LOCATION PROBLEM": "FLP",
        "AP": "AP", "ASSIGNMENT": "AP", "ASSIGNMENT PROBLEM": "AP",
        "MIXTURE": "Mixture", "OTHERS": "Others",    }
    if upper in aliases:
        return aliases[upper]
    if upper.startswith("OTHERS") or upper.startswith("SALES-BASED LINEAR PROGRAMMING"):
        return "Others"
    matches = [
        label for label, pattern in [
            ("Mixture", r"\bMIXTURE\b"),
            ("NRM", r"\bNRM\b|NETWORK REVENUE MANAGEMENT"),
            ("RA", r"\bRA\b|RESOURCE ALLOCATION"),
            ("TP", r"\bTP\b|TRANSPORTATION"),
            ("FLP", r"\bFLP\b|\bUFLP\b|FACILITY LOCATION"),
            ("AP", r"\bAP\b|ASSIGNMENT"),
            ("Others", r"\bOTHERS"),
        ]
        if re.search(pattern, upper)
    ]
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f"Cannot unambiguously normalize class: {value!r}")


def class_to_workflow_route(label):
    """Map a semantic class to an executable workflow route.
    
    Args:
        label: Semantic problem-class label.
    
    Returns:
        The route name; Mixture and Others both map to Others.
    """
    normalized = normalize_problem_class(label)
    return "Others" if normalized in {"Mixture", "Others"} else normalized


def invoke_classifier(query):
    """Retrieve FileQA once and classify once, with one shared model call."""
    query = str(query)
    observation = fileqa_retrieve(query)
    rules = prefix.split("Examples of the required ReAct process:", 1)[0]
    rules = rules.replace(
        "You must follow the ReAct workflow and call FileQA exactly once.",
        "The program has already called FileQA exactly once; its observation is supplied below.",
    )
    rules = re.sub(r"For the Action Input, provide.*?coupling mechanisms\.", "", rules, flags=re.S)
    prompt = (rules + "\nFileQA Observation (supporting references, not votes):\n" + str(observation)
              + "\nCurrent optimization problem:\n" + query
              + "\nReturn exactly ONE token: TP, NRM, RA, FLP, AP, Mixture, or Others. "
                "Do not request tools, or add Thought/Action/Observation headers or explanations.")
    response = make_llm().invoke([HumanMessage(content=prompt)])
    output = str(response.content or "").strip().strip("`")
    output = re.sub(r"^Final Answer:\s*", "", output, flags=re.I).strip()
    return {"output": output, "normalized_label": normalize_problem_class(output),
            "fileqa_calls": 1,
            "intermediate_steps": [(AgentAction(tool="FileQA", tool_input=query,
                                      log="Program-invoked FileQA"), observation)]}

## 7. Data retrieval and formulation

Python invokes the existing CSVQA Tool once before a single modeling request. Planned extraction and all route demonstrations remain. The modeling request receives the resulting Observation without an agent deciding whether to call the tool.


In [ ]:
def csvqa_prompt_view(observation, max_chars=16000):
    """Profile large prompt inputs; never change the complete runtime observation."""
    if len(observation) <= max_chars:
        return observation
    try:
        payload = json.loads(observation)
    except (TypeError, ValueError):
        return observation
    if not isinstance(payload, dict) or not isinstance(payload.get("tables"), list):
        return observation
    view = dict(payload)
    if "logical_records" in payload:
        view["logical_records"] = {
            family: records[:1]
            for family, records in payload["logical_records"].items()}
        view["_prompt_logical_record_counts"] = {
            family: len(records) for family, records in payload["logical_records"].items()}
    view["_prompt_view"] = {"records_are_samples": True,
        "runtime_variable": "CSVQA_DATA", "runtime_records_are_complete": True,
        "instruction": "Derive indices and coefficients from complete runtime records, not these samples. All _prompt_* fields are prompt-only."}
    if "logical_records" in payload:
        view["_prompt_view"]["table_record_display"] = "Record samples are grouped in logical_records; tables below retain lineage/schema only. Complete runtime tables and logical_records are unchanged."
    output_tables = []
    for table in payload["tables"]:
        records = table.get("records", [])
        current = dict(table)
        indices = sorted({i for i in [0, 1, len(records)-2, len(records)-1] if 0 <= i < len(records)})
        current["records"] = [] if "logical_records" in payload else [records[i] for i in indices]
        current["_prompt_record_count"] = len(records)
        columns = list(table.get("columns", [])[:1])
        for relationship in payload.get("relationships", []):
            for axis_name in ["row_axis", "column_axis"]:
                axis = relationship.get(axis_name, {})
                if axis.get("table_id") == table.get("table_id") and axis.get("id_column"):
                    columns.append(axis["id_column"])
            if relationship.get("matrix_table_id") == table.get("table_id") and relationship.get("row_id_column"):
                columns.append(relationship["row_id_column"])
        profiles = {}
        for column in dict.fromkeys(columns):
            values, seen = [], set()
            for record in records:
                if column not in record.get("values", {}):
                    continue
                value = record["values"][column]
                key = json.dumps(value, ensure_ascii=False, sort_keys=True)
                if key not in seen:
                    seen.add(key)
                    values.append(value)
            profiles[column] = values
        current["_prompt_identifier_values"] = profiles
        fractional_columns = []
        numeric_column_count = 0
        for column in table.get("columns", []):
            values = [record.get("values", {}).get(column) for record in records]
            values = [value for value in values if value is not None and str(value).strip()]
            if not values:
                continue
            try:
                numbers = [float(value) for value in values]
            except (TypeError, ValueError):
                continue
            if not all(math.isfinite(number) for number in numbers):
                continue
            numeric_column_count += 1
            if any(not number.is_integer() for number in numbers):
                fractional_columns.append(column)
        current["_prompt_numeric_types"] = {
            "numeric_column_count": numeric_column_count,
            "fractional_numeric_columns": fractional_columns,
            "instruction": "Types cover all returned rows, not only samples. Parse coefficients with float; identifiers and decision domains are separate."}
        output_tables.append(current)
    view["tables"] = output_tables
    rendered = json.dumps(view, ensure_ascii=False, separators=(",", ":"))
    if len(rendered) > max_chars:
        for table in output_tables:
            samples = table["records"]
            table["records"] = [samples[i] for i in sorted({0, len(samples)-1}) if samples]
        rendered = json.dumps(view, ensure_ascii=False, separators=(",", ":"))
    return rendered


FORMULATION_OUTPUT_INSTRUCTIONS = '\nReturn only a compact symbolic model and Data Mapping rules. Define each index set,\nparameter and variable once; do not enumerate coefficient values or individual rows.\nThe query, not the assigned route or an example, determines domains and constraints.\nFor a repeated daily/weekly operating schedule, cover shifts across the end/start of\nthe period with modular indices. A genuinely finite horizon instead uses the stated\ninitial conditions; do not impose a zero start on a continuously repeating schedule.\nFor a tour, choose one consistent graph representation: directed arcs use one incoming\nand one outgoing arc per node and their full directed objective; undirected edges use\none variable per unordered edge and degree two. Subtour constraints must use the same\nrepresentation. Preserve the original objective, including additive constants.\n\nWhen the query requires reciprocal payments or equal income and expenditure, derive\nboth totals from the stated source/recipient axes before writing the equality.\nIf a[i,j] is work by source j for recipient i and p[j] is its unit rate, source i earns\np[i] * sum(a[k,i] for k != i), while recipient i spends\nsum(a[i,j] * p[j] for j != i). Self-work/self-transfers must be excluded on both sides,\nor included and cancelled on both sides. A supplied total contribution replaces the\ncorresponding full column total, not the diagonal coefficient. Never use a single\nself-entry as all income. Use this balance only when the query actually requires it.\n\nAuxiliary ordering domains must allow every required distinct position. For a tour\nwith q non-anchor nodes, use ranks 1..q (or an equivalent q-position range), not 2..q.\nOne consistent directed MTZ formulation is u[i]-u[j]+q*x[i,j] <= q-1 for distinct\nnon-anchor i,j, with 1 <= u[i] <= q. The anchor is excluded from these constraints;\ndo not require rank increases on the return edge. Match domains, big-M, index ranges\nand arc representation; do not mix pieces from different example formulations.\n'

def formulate_with_csvqa(query, dataset_address, route, system_prompt, tool_description, prefix, suffix):
    """Call CSVQA once, then model its Observation once; no agent protocol repair."""
    route = normalize_route(route)
    if CSVQA_MODE_BY_ROUTE[route] == "planned":
        system_prompt = CSVQA_PLANNED_PROMPTS[route]
        tool_description = CSVQA_TOOL_DESCRIPTIONS[route]
        prefix = prefix.replace("please output required parameters in a whole text, including all vectors and matrices.",
                                "return symbolic parameters and an exact source Data Mapping; do not enumerate values.")
    llm, qa_tool, csvqa_result = build_csvqa_components(
        dataset_address, system_prompt, tool_description, route=route, user_query=query,
    )
    try:
        qa_tool.invoke(query)
    except Exception as exc:
        exc.csvqa_result = dict(csvqa_result)
        exc.pipeline_stage = "data_extraction"
        raise
    prompt = ('Reference examples demonstrate methods only. Their files, entities, numerical values,\nand constraints are not inputs to the current problem. Only the Original Query and\ncurrent source observation define this instance. Use complete query-selected source\nsets, not example entity lists or profile samples. A coefficient matrix does not by\nitself introduce another decision dimension. Do not duplicate one aggregate demand\ninto separate groups unless the query or data actually distinguishes those groups.\nFor rectangular assignment matrices, retain the stated entity roles and cardinalities;\ncaption rows are not entities and row/column orientation need not match an example.\n\n<workflow_guidance>\n'
              + prefix.replace("{{", "{").replace("}}", "}") +
              "\n</workflow_guidance>\n" +
              FORMULATION_OUTPUT_INSTRUCTIONS +
              "\nCSVQA has already been called exactly once by the program. Its complete current "
              "Observation is supplied below. Large tables may use an explicitly marked source profile; "
              "all runtime records and coefficients remain complete. Ignore sample size when defining index sets. "
              "Do not numerically solve this problem; output its mathematical formulation only. Do not "
              "request tools or copy example data. Use a concise symbolic model and an exact Data "
              "Mapping with table_id and column names. Do not enumerate numerical tables. "
              "Bind every parameter to source data or a query-defined expression; never invent a "
              "missing limit. If total capacity refers to listed per-entity capacities, retain "
              "their bounds and their sum. Preserve all variable domains, constraints, objective "
              "sense and additive constants. Define each index set from ALL current returned entities "
              "unless the original query explicitly restricts it. Demonstration names, counts and "
              "profile samples must never determine the current entity set. Optional matrix alias "
              "mappings apply only when present; otherwise preserve exact raw ID matching.\n" +
              f"<current_problem>\nOriginal Query:\n{query}\nObservation:\n{csvqa_prompt_view(csvqa_result['observation'])}\n</current_problem>")
    try:
        messages = [HumanMessage(content=prompt)]
        if CSVQA_MODE_BY_ROUTE[route] == "planned" and "logical_table_index" in json.loads(csvqa_result["observation"]):
            messages.insert(0, SystemMessage(content=INDEX_MODEL_CONTRACT))
        result = llm.invoke(messages)
    except Exception as exc:
        exc.csvqa_result = dict(csvqa_result)
        exc.pipeline_stage = "formulation"
        raise
    return {"formulation": result.content, **csvqa_result}


@lru_cache(maxsize=1)
def _load_rag_table():
    """Read the unified examples once, preserving every field as text."""
    frame = read_csv_compat(RAG_EXAMPLES_ALL_PATH, dtype=str, keep_default_na=False)
    required = {"prompt", "Data_address", "Related", "Required Data", "Label", "Code", "Type"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"RAG example CSV is missing columns: {sorted(missing)}")
    frame["Type"] = frame["Type"].map(normalize_route)
    return frame


def load_rag_examples(route):
    """Return one canonical route; UFLP maps to FLP and Mixture to Others."""
    frame = _load_rag_table()
    examples = frame.loc[frame["Type"].eq(normalize_route(route))].copy()
    if examples.empty:
        raise ValueError(f"No RAG examples found for route {route!r}")
    return examples


@lru_cache(maxsize=None)
def _get_rag_store(route):
    """Share a semantic index across modeling and code generation for each route."""
    documents = [
        Document(
            page_content=example["prompt"],
            metadata={"example_json": json.dumps(example, ensure_ascii=False)},
        )
        for example in load_rag_examples(route).to_dict("records")
    ]
    # Embed the problem only: code/formulation length must not dominate similarity.
    return FAISS.from_documents(documents, make_embeddings())


def retrieve_rag_examples(route, query, k=1):
    """Retrieve structured example dictionaries, never parse CSVLoader display text."""
    if k < 1:
        return []
    route = normalize_route(route)
    documents = _get_rag_store(route).similarity_search(
        str(query), k=min(k, len(load_rag_examples(route))),
    )
    return [json.loads(document.metadata["example_json"]) for document in documents]


def retrieve_similar_texts(query, retriever):
    # The separate query-only reference dataset still uses its original retriever.
    return [doc.page_content for doc in retriever.invoke(query)]


def escape_braces(text):
    """Protect literal example braces when embedding text into an agent template."""
    return text.replace("{", "{{").replace("}", "}}")


def rag_example_observation(example):
    """Prefer the curated evidence, including its original source-row identifiers."""
    evidence = example["Required Data"].strip()
    if evidence:
        return evidence
    blocks = []
    for raw in example["Data_address"].splitlines():
        if raw.strip():
            frame = read_csv_compat(raw.strip(), dtype=str, keep_default_na=False)
            blocks.append(f"Source: {raw.strip()}\n{frame.to_json(orient='records', force_ascii=False)}")
    return "\n\n".join(blocks)


def build_formulation_examples(route, query, k=1):
    """Render the common ReAct example format once for all CSV modeling routes."""
    examples = []
    for example in retrieve_rag_examples(route, query, k):
        block = f"""Question: Based on the following problem description and data, formulate a complete mathematical model. {example['prompt']}

Thought: Retrieve all data needed by the question, in source order. Preserve identifiers, coefficients, and any additional constraints. If the question does not request a subset, retrieve the whole dataset.

Action: CSVQA

Action Input: Retrieve all data {example['Related'].strip()} required to formulate this model. Preserve source order and exact identifiers and values, without simplification or abbreviation.

Observation: {rag_example_observation(example)}

Thought: Use the retrieved data as parameters and include all query-specific constraints. Choose variable domains from the question. Return the complete mathematical model in markdown, with $ or $$ for mathematical expressions and without extra explanations or LaTeX environments.

Final Answer:
{example['Label']}"""
        # Prefix is itself a PromptTemplate; escape the entire block exactly once.
        examples.append(escape_braces(block))
    return "\n\n".join(examples)


def get_NRM_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("NRM", query, k=1)
    csvqa_system_prompt = CSVQA_PLANNED_PROMPTS["NRM"]
    csvqa_tool_description = CSVQA_TOOL_DESCRIPTIONS["NRM"]

    route_instruction = """Call CSVQA exactly once and return an ABSTRACT model. Define
index sets, parameters, variables, objective, and constraints symbolically. Include a short Data
Mapping that identifies every source by exact table_id and exact column name. Do not copy record
values or literal record counts."""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "NRM", csvqa_system_prompt, csvqa_tool_description, prefix, suffix,
    )


## 8. Resource allocation

Build the route-specific formulation using CSV evidence and retrieved examples.


In [ ]:
def get_RA_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("RA", query, k=3)
    csvqa_system_prompt = CSVQA_PLANNED_PROMPTS["RA"]
    csvqa_tool_description = CSVQA_TOOL_DESCRIPTIONS["RA"]

    route_instruction = """You MUST call CSVQA exactly once before the Final Answer and use all returned rows.
Keep the supplied file order and original row order. Never sort or reset an index.
Use an explicit business ID column when present; never synthesize business IDs from row numbers.
Do not infer a file's semantic role only from its filename or position.
Align profits, resource consumption, and capacities through explicit column/ID keys.
Every resource-capacity dimension must receive a capacity constraint, and each
activity must appear in the objective and all applicable resource constraints.
Infer integer/binary/continuous domains from the user description and data.
A requested count of indivisible objects, people, properties or trips is nonnegative integer,
including how-many decisions; binary choices remain binary. Ordinary production/flow amounts
are continuous unless whole/integer quantities are required. The number of product types or
resource locations defines an index-set size, not integrality of the production amount.
Distinguish a global activity decision from separate resource-by-activity allocations. When independent
warehouses, machines or other resources can each allocate to activities, use decisions indexed by
both resource and activity; do not constrain one identical global expression by every resource's limit.
Each resource-indexed capacity constraint must depend on that resource through its decisions or
consumption coefficients. For different resource dimensions consumed by one global activity, retain
global activity variables and the dimension-specific consumption coefficients instead.
Never invent a total capacity, default RHS, or omitted consumption coefficient.
Return a concise ABSTRACT mathematical model with symbolic index sets, parameters, variables,
objective and constraints. Include a Data Mapping for every parameter using exact table_id and
column names from CSVQA_DATA. Keep every resource dimension and original business identifier.
Do not enumerate record values or hard-code entity counts; Python supplies the source records.
Ignore unrelated administrative columns; numeric columns are not automatically resources or costs.
"""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "RA", csvqa_system_prompt, csvqa_tool_description, prefix, suffix,
    )


## 9. Transportation

Build the route-specific formulation using CSV evidence and retrieved examples.


In [ ]:
def get_TP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("TP", query, k=3)
    csvqa_system_prompt = CSVQA_LEGACY_SYSTEM_PROMPT
    csvqa_tool_description = CSVQA_LEGACY_TOOL_DESCRIPTION

    route_instruction = """Call CSVQA at least once and return a complete numerical formulation. Retrieved
Information must contain every identifier and coefficient required by downstream code. Preserve
source order and do not invent, aggregate, or silently omit data."""
    prefix = f"""You are an assistant that formulates mathematical optimization models.

    The examples below illustrate model structure only; never copy their data.

    {few_shot_examples}

    {route_instruction}
    Use the user query as the authority for objective sense, variable domains, boundary semantics,
    and additional constraints. Do not invent missing data or silently omit returned data.
    """

    suffix = """

    Begin!

    User Description: {input}
    {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "TP", csvqa_system_prompt, csvqa_tool_description, prefix, suffix,
    )


## 10. Assignment

Build the route-specific formulation using CSV evidence and retrieved examples.


In [ ]:
def get_AP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("AP", query, k=1)
    csvqa_system_prompt = 'Retrieve the documents in order from top to bottom. Use the retrieved context to answer the question. If mention a certain kind of product, retrieve all the relavant product information detail judging by its product name. If not mention a certain kind of product, retrieve all the data instead.Context: {context}'
    csvqa_tool_description = 'Use this tool to answer Querys based on the provided CSV data and retrieve product data similar to the input query.'

    prefix = f"""You are an assistant that generates a mathematical model based on the user's description and provided CSV data.

            Please refer to the following example and generate the answer in the same format:

            {few_shot_examples}

            Note: Please retrieve all neccessary information from the CSV file to generate the answer. When you generate the answer, please output required parameters in a whole text, including all vectors and matrices.

            When you need to retrieve information from the CSV file, use the provided tool.

            """

    suffix = """

            Begin!

            User Description: {input}
            {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "AP", csvqa_system_prompt, csvqa_tool_description, prefix, suffix,
    )


## 11. Facility location

Build the route-specific formulation using CSV evidence and retrieved examples.


In [ ]:
def get_FLP_response(query, dataset_address):
    few_shot_examples = build_formulation_examples("FLP", query, k=1)
    csvqa_system_prompt = ('Retrieve the documents in order. Use the given context to answer the question. If mention a certain kind of product, retrieve all the relavant product information detail judging by its product name. If not mention a certain kind of product, make sure that all the data is retrieved.Context: {context}' + ' Preserve every facility ID, customer ID, FixedCost, Capacity, Demand, and cost-matrix axis together with its source-row position. FixedCost and Capacity must use the same facility ID; the matrix row and column IDs must explicitly match facilities and customers, with its source orientation and shape retained. Model a two-dimensional shipment decision only. Use a stated capacity when present; an absent capacity is unresolved evidence, not zero. Do not transpose, truncate, pad, zero-fill, or infer an extra commodity/product axis.')
    csvqa_tool_description = 'Use this tool to answer Querys based on the provided CSV data and retrieve product data similar to the input query.'

    prefix = f"""You are an assistant that generates a mathematical model based on the user's description and provided CSV data.

            Please refer to the following example and generate the answer in the same format:

            {few_shot_examples}

            Note: Please retrieve all neccessary information from the CSV file to generate the answer. When you generate the answer, please output required parameters in a whole text, including all vectors and matrices.

            When you need to retrieve information from the CSV file, use the provided tool.

            """

    suffix = """
            Begin!

            User Description: {input}
            {agent_scratchpad}"""

    return formulate_with_csvqa(
        query, dataset_address, "FLP", csvqa_system_prompt, csvqa_tool_description, prefix, suffix,
    )


## 12. Other and mixed problems

CSV cases use full-file statistics and query-name evidence alongside the first 10 rows, then generate an Abstract Model Plan and code that reads complete files. Evidence does not select rows automatically. Query-only modeling keeps the original notebook workflow.


In [ ]:
def build_few_shot_Other(examples, t="Model"):
    """Render already-structured examples; inserted prompt values need no brace escaping."""
    field = "Label" if t == "Model" else "Code"
    thought = (
        "I need to create an Abstract Model Plan based on the query and CSV schema."
        if t == "Model" else
        "I need to generate a single, complete, executable Gurobi Python code block."
    )
    return "\n\n".join(
        f"<EXAMPLE>\nQuestion: {example['prompt']}\nThought: {thought}\n"
        f"Final Answer:\n{example[field]}\n</EXAMPLE>"
        for example in examples if example[field].strip()
    )


def csv_schema_preview(dataset_address, query=""):
    """Preview plus full-file evidence; matching never chooses or filters the model data."""
    normalize = lambda value: re.sub(r"\s+", " ", str(value)).strip().casefold()
    normalized_query = normalize(query)
    quoted = re.findall(r'"([^"\n]+)"|“([^”\n]+)”|‘([^’\n]+)’|(?<!\w)\x27([^\x27\n]+)\x27(?!\w)', query)
    names = {normalize(next(part for part in group if part)) for group in quoted}
    blocks = []
    for raw in normalize_data_address(dataset_address).splitlines():
        path = Path(raw).expanduser()
        path = path if path.is_absolute() else PROJECT_ROOT / path
        typed = read_csv_compat(path)
        frame = read_csv_compat(path, dtype=str, keep_default_na=False)
        stats, columns, terms = {}, {}, set(names)
        for column in frame:
            values = frame[column]
            normalized = values.map(normalize)
            columns[column] = normalized
            present = values[normalized.ne("")]
            numbers = pd.to_numeric(present, errors="coerce")
            stats[column] = {"missing": int(normalized.eq("").sum()), "unique_nonempty": int(present.nunique())}
            if len(present) and numbers.notna().all():
                stats[column]["numeric_range"] = [float(numbers.min()), float(numbers.max())]
            # Unquoted evidence must literally name a real column or complete nonnumeric value.
            candidates = [column] + present.unique().tolist()
            for value in candidates:
                term = normalize(value)
                if term and not re.fullmatch(r"[-+\d.,%]+", term) and re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", normalized_query):
                    terms.add(term)
        evidence = []
        for term in sorted(terms):
            matches = []
            for column, values in columns.items():
                exact, prefix, contains = values.eq(term), values.str.startswith(term), values.str.contains(term, regex=False)
                if contains.any():
                    matches.append({"column": column, "exact": int(exact.sum()), "prefix": int(prefix.sum()),
                                    "contains": int(contains.sum()), "examples": frame.loc[contains, column].drop_duplicates().head(3).tolist()})
            evidence.append({"term": term, "matching_columns": matches,
                             "exact_matching_columns": sum(item["exact"] > 0 for item in matches)})
        blocks.append(
            f"File: {path.resolve()}\nCSV delimiter: comma; first line consumed as header.\nTotal rows: {len(frame)}\nColumns: {list(frame.columns)}\n"
            f"Parsed column types: {typed.dtypes.astype(str).to_dict()}\n"
            f"Preview only (first 10 rows):\n{frame.head(10).to_string(index=False)}\n"
            f"Full-file column statistics: {json.dumps(stats, ensure_ascii=False)}\n"
            "Matching uses casefold and collapsed/trimmed whitespace; original IDs and leading zeros are preserved.\n"
            "Counts are per column and per individual condition, not intersections. Empty matching_columns means zero matches in every column.\n"
            f"Query-name evidence: {json.dumps(evidence, ensure_ascii=False)}"
        )
    if not blocks:
        raise ValueError("No external CSV paths were supplied")
    return "\n\n---\n\n".join(blocks)


def get_Others_response(user_query: str, dataset_address: str):
    if CSVQA_MODE_BY_ROUTE["Others"] == "planned":
        few_shot_examples = build_formulation_examples("Others", user_query, k=3)
        prefix = ("Generate a mathematical optimization model for the current query and its source data. "
                  "Infer the model type, variable domains, and constraints from the current query, not from "
                  "a demonstration. Preserve the full objective, all restrictions, and an exact Data Mapping.\n"
                  + few_shot_examples + "\n" + ALIGNMENT_MODEL_HINT)
        return formulate_with_csvqa(
            user_query, dataset_address, "Others", CSVQA_PLANNED_PROMPTS["Others"],
            CSVQA_TOOL_DESCRIPTIONS["Others"], prefix, "",
        )
    examples = retrieve_rag_examples("Others", user_query, k=3)
    llm2 = make_llm()

    try:
        print("\n[Gurobi Pipeline Step 1/3]: Getting CSV Scheme...")

        schema = csv_schema_preview(dataset_address, query=user_query)
        schema += "\n" + ALIGNMENT_MODEL_HINT
        schema += ("\nDerive each decision index set from exact current CSV entity identifiers. "
                   "Never replace source IDs with a guessed numeric range or demonstration IDs. "
                   "Distinguish axis captions and scalar-parameter rows from decision-entity rows "
                   "by their labels and declared roles, not their positions. Preserve scalar "
                   "parameters separately; do not add them to matrix coefficient sums. "
                   "Query examples are not exhaustive entity lists. Infer matrix row and column "
                   "roles independently and use every relevant source entity.")
        if re.search(r"\bsymmetric\b", user_query, re.I):
            schema += ("\nWhen the query explicitly states symmetry, align matrix axes by entity ID. "
                       "Fill a missing off-diagonal entry from its present transpose; preserve zeros. "
                       "Reject pairs with both entries missing or conflicting values beyond numeric tolerance. "
                       "Without explicit symmetry, preserve direction and never mirror or zero-fill entries.")
        if re.search(r"\bshut(?:down| down)\b", user_query, re.I):
            schema += "\nMinimum-down-time restrictions activate only on an on-to-off transition; down-time constraints alone must never forbid an always-on sequence."
        if "\\sum" in user_query:
            schema += "\nPreserve the query's explicit summation scope: aggregate every summed index inside the expression; never replace a summed index with a separate family of constraints."
        if re.search(r"\bworkers?\b", user_query, re.I) and re.search(r"\btasks?\b", user_query, re.I):
            schema += "\nIn a worker-task matrix, distinguish embedded row/column-axis captions from actual workers. Select workers by their identifiers, exclude label rows, then validate worker counts; never truncate by position."
        print("\n[Gurobi Pipeline Step 2/3]: Constructing Abstract Model...")

        few_shot_block_abstract = build_few_shot_Other(examples, t='Model')
        print(f"[Few-Shot Abstract Examples]:\n{few_shot_block_abstract}...")
        
        abstract_model_template = """
You are an expert optimization modeler.
Your task is to create an "Abstract Model Plan" based on the user's query and the CSV Schema (data structure).
This plan is *not* Gurobi code or mathematical formulas, but a clear, step-by-step reasoning process in English.

[Examples]
{few_shot_examples}

[Current Task]
User Query: {query}

CSV Schema:
{schema}

An explicitly enumerated complete entity set in the query is authoritative: extra CSV entities
must not enlarge that set unless the query explicitly delegates its definition to the data.
For a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial
binary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous
level or a first-period ramp limit unless the query explicitly supplies that continuous boundary.
For recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period
when a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.
State and implement the original objective sense and complete value expression, including constants.
Do not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.
Examples illustrate style; the current query and schema determine the actual scope, indices and boundaries.

Data selection: state for each entity table whether all rows or a subset is required; name the
filter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.
Without a query-requested filter, use the complete entity set. A preview is never the selected set.
Matching statistics are evidence, not instructions: do not broaden a zero-match condition to
substring matching or all rows. Check selected keys and applicable counts at runtime; recompute
combined conditions rather than treating individual counts as intersection counts.

[Your Output]
You must strictly follow this format for your "Abstract Model Plan" output:

[Abstract Model Plan START]
1.  **Analyze Query:** The user wants to {{{{Your analysis}}}}.
2.  **Identify Model Type:** Based on the query, this is a {{{{e.g., LP, MIP, Fixed-Charge, Blending}}}} problem.
3.  **Define Index Sets:** The primary indices are {{{{e.g., Products, Workers, Sources, Destinations}}}}.
4.  **Define Decision Variables:**
    -   `x[i]` = {{{{Describe first variable, e.g., 'quantity of product i'}}}}. Type: {{{{GRB.CONTINUOUS / GRB.INTEGER}}}}.
    -   `y[i]` = {{{{Describe second variable, e.g., 'if product i is produced'}}}}. Type: {{{{GRB.BINARY}}}}.
5.  **Identify Parameters (from Schema):**
    -   Objective coefficients (e.g., profit) will come from column(s): {{{{e.g., 'Price', 'Production Cost'}}}}.
    -   Constraint coefficients (e.g., resource use) will come from: {{{{e.g., 'Resource 1', 'Available Time'}}}}.
    -   Constraint RHS (limits) will come from: {{{{e.g., 'Max Demand'}}}}.
6.  **Formulate Objective:** {{{{Describe objective, e.g., 'Maximize sum((schema['Price'] - schema['Cost']) * x[i] - schema['FixedCost'] * y[i])'}}}}.
7.  **Formulate Constraints:**
    -   Constraint 1 (e.g., Resource Limit): {{{{Describe constraint 1, e.g., 'sum(schema['Resource 1'][i] * x[i]) <= schema['Available']'}}}}.
    -   Constraint 2 (e.g., Linking): {{{{Describe constraint 2, e.g., 'x[i] <= M * y[i])'}}}}.
    -   ... (Other constraints) ...
[Abstract Model Plan END]
"""
        abstract_prompt = PromptTemplate(
            template=abstract_model_template,
            input_variables=["query", "schema", "few_shot_examples"]
        )
        
        abstract_model_chain = LLMChain(llm=llm2, prompt=abstract_prompt)
        
        abstract_model_plan = abstract_model_chain.run(
            query=user_query,
            schema=schema,
            few_shot_examples=few_shot_block_abstract
        )
        print(f"[Observation]:\n{abstract_model_plan}")
        
        print("\n[Gurobi Pipeline Step 3/3]: Generating Gurobi Code...")
        
        few_shot_block_code = build_few_shot_Other(examples, t='Code')
        
        code_gen_template = """
You are an expert Gurobi programmer.
Your task is to strictly follow the User Query, CSV Schema, and "Abstract Model Plan" to translate them into a single, complete, executable Gurobi Python code block.
The code must start with ```python and end with ```.
The code must include the necessary imports from gurobipy, pandas, numpy, re, math or json. Do not import sys or os.
Create exactly one Gurobi model in m or model and call its optimize() exactly once, both at top level. Do not wrap model creation or solving in functions or try/except; let errors propagate to the execution harness. Do not write files or call computeIIS().
The code must read all supplied CSV files needed by the model, at these exact newline-separated paths:
{dataset_address}
Read their complete contents at runtime. The schema preview is not the full dataset.
Preserve explicit business identifiers and matrix axes; do not invent filenames or columns.
Data-reading contract: these files were parsed with pandas.read_csv(path, sep=',').
The displayed preview is space-aligned for readability; it does not imply a whitespace delimiter.
pandas already consumes the CSV header. Select entity and parameter rows through identifier values
or validated predicates, never guessed iloc offsets or a guessed number of header rows.
Normalize identifier types explicitly on both sides of every lookup/join, including matrix row
indices versus column labels. Preserve the actual identifiers and derive dimensions from all rows.
Series.apply receives one cell value, not a row; use vectorized column predicates or DataFrame.apply
with axis=1 when a predicate needs a row. Every comprehension must use the indices it binds, not
stale variables from a preceding loop. Do not silently skip failed conversions or missing coefficients.
Use setObjective for the exact metric requested by the query, including all additive constants,
signs and units. An optimizer-equivalent objective is not an equivalent reported objective value;
do not drop constant terms and then merely restore them in printed output.
The code must *fully* implement all variables, objectives, and constraints from the "Abstract Model Plan".

[Examples of the Full Process]
{few_shot_examples}

[User Query]
{query}

[CSV Schema]
{schema}

[Abstract Model Plan]
{abstract_plan}

An explicitly enumerated complete entity set in the query is authoritative: extra CSV entities
must not enlarge that set unless the query explicitly delegates its definition to the data.
For a one-off finite-horizon schedule, change/ramp constraints compare consecutive modeled periods. An initial
binary on/off state is not a pre-horizon continuous quantity: do not invent an initial continuous
level or a first-period ramp limit unless the query explicitly supplies that continuous boundary.
For recurring daily/cyclic operation, preserve wrap-around coverage across the last/first period
when a working shift spans the boundary; do not impose a fictitious empty start to a recurring day.
State and implement the original objective sense and complete value expression, including constants.
Do not offer sign-reversed or constant-shifted optimizer-equivalent objectives as alternatives.
Examples illustrate style; the current query and schema determine the actual scope, indices and boundaries.

Data selection: state for each entity table whether all rows or a subset is required; name the
filter column, exact condition, entity key, and joins. Preserve shared resource/parameter tables.
Without a query-requested filter, use the complete entity set. A preview is never the selected set.
Matching statistics are evidence, not instructions: do not broaden a zero-match condition to
substring matching or all rows. Check selected keys and applicable counts at runtime; recompute
combined conditions rather than treating individual counts as intersection counts.

Reserve m and model for the Gurobi model; never reuse them as loop variables or other values.
Keep business identifiers as Python dictionary keys, separate from Gurobi names.
Use name='' for addVars/addConstrs so business keys are not expanded into solver names;
use only short ASCII names for individually named variables and constraints.
Never invent, truncate, or replace required coefficients with dummy or random data.
Before optimization, validate coefficient dimensions and identifier coverage against decision index sets;
raise an explicit error for missing data instead of guessing, padding, or silently dropping entries.

[Your Gurobi Code]
"""

        code_gen_template += "\n" + CSV_SOLVER_INSTRUCTIONS + MODEL_INTERFACE_INSTRUCTIONS

        code_gen_prompt = PromptTemplate(
            template=code_gen_template,
            input_variables=["query", "schema", "abstract_plan", "dataset_address", "few_shot_examples"]
        )
        
        code_gen_chain = LLMChain(llm=llm2, prompt=code_gen_prompt)
        
        final_code = code_gen_chain.run(
            query=user_query,
            schema=schema,
            abstract_plan=abstract_model_plan,
            dataset_address=dataset_address,
            few_shot_examples=few_shot_block_code 
        )
        
        if "```python" in final_code:
            code_block = final_code.split("```python", 1)[1]
            if "```" in code_block:
                code_block = code_block.split("```", 1)[0]
            final_answer = "```python\n" + code_block.strip() + "\n```"
        else:
            if not final_code.strip().startswith("import"):
                print(f"[Warning] Step 3 output was not a valid code block. Output: {final_code[:200]}...")
                final_answer = f"Error: Code generation failed. LLM returned non-code output:\n{final_code}"
            else:
                print("[Warning] Step 3 output missed ```python tag, adding it.")
                final_answer = "```python\n" + final_code.strip() + "\n```"
            
        print(f"[Final Answer]:\n{final_answer}")
        return {
            "formulation": abstract_model_plan, "code": final_answer, "observation": schema,
            "trace": {"status": "LEGACY_SCHEMA", "planner_attempt_count": 0,
                      "payload_hash": hashlib.sha256(schema.encode()).hexdigest()},
        }

    except Exception as e:
        print(f"[Gurobi Pipeline Error]: {e}")
        raise

def get_others_without_CSV_response(query):
    llm = make_llm()

    loader = CSVLoader(file_path=str(PROJECT_ROOT / "Large_Scale_Or_Files/RAG_Example_Others_Without_CSV.csv"), encoding="utf-8")
    documents = loader.load()

    embeddings = make_embeddings()
    vectors = FAISS.from_documents(documents, embeddings)

    retriever = vectors.as_retriever(search_kwargs={'k': 5})
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
    )

    qa_tool = Tool(
        name="ORLM_QA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer Querys."
            "Provide the Query as input, and the tool will retrieve the relevant information from the file and use it to answer the Query."
        ),
    )

    few_shot_examples = []
    similar_results = retrieve_similar_texts(query, retriever)

    for content in similar_results:

        split_at_formulation = content.split("Data_address:", 1)
        problem_description = split_at_formulation[0].replace("prompt:", "").strip()
        split_at_address = split_at_formulation[1].split("Label:", 1)
        split_at_label = split_at_address[1].split("Related:", 1)
        label = split_at_label[0].strip() 

        label = label.replace("{", "{{").replace("}", "}}")
        example = (
            "<EXAMPLE>\n"
            f"Question: {problem_description}\n\n"
            "Thought: Read the question and 1) identify the goal (minimize time/cost/crew or maximize throughput/value) and collect per-unit coefficients; 2) define decision variables and pick domains: counts of trips/units/vehicles are nonnegative integers, yes or no choices are binary, divisible flows or weights are nonnegative reals; 3) write the linear objective from the coefficients; 4) add constraints in this order: demand or target (exactly, at least, at most), capacity or supply upper bounds, flow conservation or stage linking across nodes or arcs, and any share or ratio limits rewritten as linear inequalities using the given per-unit rates, plus any minimum or maximum usage; 5) add nonnegativity and the chosen integrality or binary domains; 6) output only the LP: objective first, then constraints line by line with brief labels if needed, then a final line stating the variable domains."
            "Final Answer:\n"
            f"{label}\n"
            "</EXAMPLE>"
        )

        example = example.replace("{", "{{").replace("}", "}}")
        few_shot_examples.append(example)

    prefix = (
    f"{few_shot_examples}\n\n"
    """ 
    Use the following triggers to identify problem structures and apply the corresponding mathematical formulations.
    (1) INTEGER trigger:
    Question: A factory can run two machine types. How many of each machine should be installed given budget and space limits? Maximize output.
    Final Answer: 
    $\\max\\; p_1 x_1 + p_2 x_2$
    $\\text{{s.t. }} a_1 x_1 + a_2 x_2 \\le B,\\; s_1 x_1 + s_2 x_2 \\le S$
    $x_1, x_2 \\in \\mathbb{{Z}}_+.$
    (2) MULTI-PERIOD FLOW trigger:
    Question: Multi-period production with inventory and backorders. Costs for production/holding/backorder. Initial and terminal conditions given.
    Final Answer:
    \textbf{{Indices: }} t\\in T=\\{{1,\\dots,n\\}}. \\
    \textbf{{Given: }} d_t,\\ I_0,\\ B_0,\\ \\dots \\
    \textbf{{Vars: }} x_t\\ge0,\\ I_t\\ge0,\\ B_t\\ge0. \\
    \\min \\sum_t (c x_t + h I_t + p B_t) \\
    \text{{s.t. }} I_t - B_t = I_{{t-1}} - B_{{t-1}} + x_t - d_t,\\ \forall t \\
    I_n \\ge I^{{\\min}},\\ B_n=0.
    (3) LOGIC+BINARY trigger:
    Question: Choose exactly one option from set P and at least K items from set V, with quantities and budget.
    Final Answer:
    \textbf{{Sets: }} i\\in P,\\ j\\in V. \\ \textbf{{Vars: }} q_i,q_j\\ge0;\\ y_i\\in\\{{0,1\\}}, z_j\\in\\{{0,1\\}}.\\
    \\max \\sum_j f_j q_j \\
    \text{{s.t. }} \\sum_i y_i = 1,\\ \\sum_j z_j \\ge K \\
    0\\le q_i \\le M_i y_i,\\ \forall i;\\ \\ 0\\le q_j \\le M_j z_j,\\ \forall j \\
    \text{{Budget: }} \\sum_i c_i q_i + \\sum_j c_j q_j \\le B.
    """
    "USER QUESTION:\n{input}\n\n"
    "TASK:\n"
    "- Produce a complete LaTeX optimization model using ONLY information in the question.\n"
    "- Use a fixed structure INSIDE LaTeX: Indices/Sets; Given Parameters (convert all tables to arrays); Decision Variables (with domains); Objective; Constraints; Domain lines.\n"
    "- For multi-period problems, you MUST include state-balance recurrences and initial/terminal conditions explicitly.\n"
    "- Avoid nonlinear forms when possible: rewrite ratios/logic using linear constraints + binaries (big-M) with clearly defined M.\n\n"
    "You should decide VARIABLE–TYPE first!\n (If not mentioned or ambiguous, integer by default!)"
    "### FIRST RESPONSE FORMAT (exactly 3 lines) ###\n"
    "Thought: <brief>\n"
    "Action: ORLM_QA\n"
    "Action Input: {input}\n\n"
    "Begin."
)

    suffix = (
    "\n### AFTER OBSERVATION ###\n"
    "Respond with exactly two lines:\n"
    "Thought: <variable types: integer/binary/continuous>\n"
    "Final Answer: <ONLY LaTeX model. Must include: (i) indices/sets, (ii) parameter definitions (tables->arrays), (iii) explicit domain lines for EVERY variable, (iv) initial/terminal conditions if any. No prose.>\n"
    "Do NOT output anything else."
)

    agent = initialize_agent(
        tools=[qa_tool],
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        agent_kwargs={
            "prefix": prefix,
            "suffix": suffix,
            "input_variables": ["input"]
        },
        verbose=True,
        handle_parsing_errors=False,  # Enable error handling
    )

    openai.api_request_timeout = 60  
    query = query.replace('{','{{').replace('}','}}')
    output = agent.run({"input": query})

    return output


## 13. Generate Python code

get_code preserves the original query-only workflow. get_csv_code uses structured NRM data or the numerical formulation from CSVQA. Others with CSV already generates code in section 12. All prompt examples are defined in this notebook.


In [ ]:
NATIVE_CODE_SYSTEM = "Generate complete executable Python from the current Query, draft formulation,\nand runtime CSVQA_DATA. The Query determines semantics; reconcile contradictions in\nthe draft, rather than compiling an impossible or weakened version of the Query.\nUse import gurobipy and from gurobipy import GRB, and the fully qualified gurobipy API.\nRead CSVQA_DATA directly; never redefine it, copy its data into code, read external\nfiles, or assume custom helpers. Profiles contain samples, not runtime dimensions.\nDerive complete identifiers and coefficients from current Data Mapping and every\nquery-designated source fragment. Keep matrix axes, genuine eligibility restrictions,\nblank-cell semantics, and exact source identities. Do not copy example data or defaults.\nRetain variable objects under the same scalar/tuple keys used to create them; keep\nloop indices, numeric parameters and variable containers distinct. Preserve the\nQuery's objective sense, coefficients and constants. Expose a live completed model\nas module-level m; a builder must return its model and be called. Set MIPGap=1e-6,\ncall m.optimize(), do not dispose m or hide exceptions, and return complete code only.\n"
TREE_CODE_PATTERN = "For the Query's artificial single-commodity connectivity flow on an undirected\nspanning tree, the root sends n-1 units and every other node receives one. Thus\noutgoing - incoming = n-1 at the root and -1 elsewhere; all node balances sum to zero.\nDo not reverse one side alone. Every directed arc links to its original undirected\nedge with capacity (n-1) * edge_selected. Select exactly n-1 original edges.\nThe following is a structural API example, not supplied runtime objects. Construct\nnode_ids, edge_keys, directed_endpoints, edge_of_arc and root from the current records\nand Data Mapping. Preserve independent source edge identity, including parallel edges.\nm = gurobipy.Model()\nedge_selected = {edge_key: m.addVar(vtype=GRB.BINARY, name='') for edge_key in edge_keys}\nflow = {arc_key: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name='') for arc_key in directed_endpoints}\noutgoing = {node: [] for node in node_ids}\nincoming = {node: [] for node in node_ids}\nfor arc_key, endpoints in directed_endpoints.items():\n    tail, head = endpoints\n    outgoing[tail].append(flow[arc_key])\n    incoming[head].append(flow[arc_key])\n    m.addConstr(flow[arc_key] <= (len(node_ids)-1) * edge_selected[edge_of_arc[arc_key]])\n# Use ONE outgoing-minus-incoming convention for every node.\n# Construct all demands first; replace only the root entry with its supply.\nnet_supply = {node: -1 for node in node_ids}\nnet_supply[root] = len(node_ids) - 1\nassert sum(net_supply.values()) == 0\nfor node in node_ids:\n    m.addConstr(gurobipy.quicksum(outgoing[node]) - gurobipy.quicksum(incoming[node]) == net_supply[node])\nm.addConstr(gurobipy.quicksum(edge_selected.values()) == len(node_ids)-1)\n# Continue with the actual objective, parameter settings and optimize; this is not a stub.\n# Lists built by arc endpoints avoid rebinding a fixed root/node in a comprehension.\n"
FRAGMENT_CODE_PATTERN = "The Query explicitly distributes source rows across files. The supplied schema\nmetadata identifies possible physical fragments, not new decision types or extra joins.\nFor each parameter, bind ALL exact table_ids designated by its current Data Mapping\nbefore constructing its index domain or validating coefficient coverage. A prefix\ntest for only the first file is not all fragments. Do not merge unrelated parameters\nmerely because schemas match, restore excluded entities, or turn blank forbidden\nmatrix cells into zero costs. Preserve the Query's worker/item eligibility and axes.\nThis structural example is not a custom API. Define mapped_fragment_ids locally from\nthe current Data Mapping and metadata, not reference filenames or invented aliases:\ntables_by_id = {table['table_id']: table for table in CSVQA_DATA['tables']}\nfor fragment_id in mapped_fragment_ids:\n    for record in tables_by_id[fragment_id]['records']:\n        values = record['values']\n        source_identity = (fragment_id, record['source_row'])\n        # Populate the actual parameter here from its mapped columns, across ALL fragments.\n# Business keys come from mapped values; source_identity is only for independent record identity.\n# Keep metadata and loop names separate from retained solver-variable dictionaries.\n"
"""Select codegen examples from explicit query semantics and source schema only."""
import re

def _native_codegen_focus(query, payload):
    if not isinstance(payload, dict) or 'logical_records' in payload:
        return None
    query = re.sub('\\s+', ' ', str(query)).casefold()
    tables = [t for t in payload.get('tables', []) if isinstance(t, dict) and t.get('table_id') and t.get('columns')]
    normalized = {t['table_id']: {re.sub('[^a-z0-9]', '', str(c).casefold()) for c in t['columns']} for t in tables}
    node_tables = [t['table_id'] for t in tables if normalized[t['table_id']] & {'node', 'nodeid', 'site', 'siteid', 'vertex', 'vertexid'}]
    endpoint_pairs = [{'node1', 'node2'}, {'fromnode', 'tonode'}, {'source', 'target'}, {'from', 'to'}]
    edge_tables = [t['table_id'] for t in tables if any((pair <= normalized[t['table_id']] for pair in endpoint_pairs))]
    if 'spanning tree' in query and 'root' in query and ('flow' in query) and re.search('\\bundirected\\b', query) and ('connectivity' in query or re.search('single[- ]commodity', query)) and node_tables and edge_tables:
        return {'kind': 'rooted_connectivity_tree', 'node_table_ids': node_tables, 'edge_table_ids': edge_tables, 'notice': 'Schema evidence only. Derive exact root, node/edge identities, costs and domains from current Data Mapping and records.'}
    if not re.search('\\b(?:distributed|split|spread|partitioned|sharded)\\b[^.;!?]{0,160}\\bfiles?\\b', query):
        return None
    by_columns = {}
    for table in tables:
        columns = tuple((str(c) for c in table['columns']))
        if len(columns) < 2:
            continue
        by_columns.setdefault(columns, []).append(table['table_id'])
    groups = []
    for columns, ids in by_columns.items():
        if len(ids) < 2:
            continue
        keys = [c for c in columns if re.sub('[^a-z0-9]', '', c.casefold()).endswith('id') and re.search('(?<!\\w)' + re.escape(c.casefold()) + '(?!\\w)', query)]
        if keys:
            groups.append({'table_ids': ids, 'columns': list(columns), 'query_named_key_columns': keys})
    if groups:
        return {'kind': 'explicit_query_source_fragments', 'same_schema_candidates': groups, 'notice': 'Candidate fragment groups, not new entities or mandatory joins. Combine only fragments designated for one parameter by the Query/Data Mapping. Preserve rows, source_row and genuine scope/eligibility.'}
    return None

FOCUSED_CODE_SYSTEM = "Generate a complete executable Python program for the current Query, formulation,\nand runtime CSVQA_DATA. The Query determines semantics; reconcile any draft formulation\nthat contradicts it. Reference-example entities, coefficients and constraints are absent.\nUse import gurobipy and from gurobipy import GRB. Expose the completed gurobipy.Model\nas module-level m, including calling a builder if used. Read CSVQA_DATA directly; never\nredefine it, print/copy its records into code, read external files, or assume custom APIs.\nUse ordinary dictionaries of retained addVar objects with exactly the actual scalar or\ntuple business key. Keep parameter dictionaries, loop names and variable containers distinct.\nUse numeric eligibility bounds, linear zero-or-lot links, and the Query's objective signs.\nAn unconditional category quantity bound stays unconditional. A once-per-category fee\nuses OR activation: each member selection <= activation <= sum(member selections).\nRelations do not create entities. An excluded prerequisite disables its existing dependent;\nbonus indicators exist in both links and objective only when both endpoints are in-domain.\nFollow the supplied prepared-unit notice without a second conversion. No missing-data\nzero filling, guessed aliases, dummy coefficients, pass stubs or saved-code repair.\nSet MIPGap=1e-6 and call m.optimize(). Keep m live: do not dispose it or hide exceptions.\nReturn complete code only; preserve objective sense and the full expression including constants.\n"
FOCUSED_API_PATTERNS = "These are API patterns, not extra variables or business constraints. Derive all names,\ndomains and coefficients from this Query and effective source records. Do not assume any\nhelper functions or the example identifiers exist at runtime.\n# Scalar and pair keys are ordinary dict keys, avoiding addVars key flattening.\nm = gurobipy.Model()\nquantity = {item: m.addVar(vtype=GRB.INTEGER, name='') for item in item_ids}\nselected = {item: m.addVar(vtype=GRB.BINARY, name='') for item in item_ids}\npair_active = {}\nfor pair_key, coefficient in bonus_coefficients.items():\n    left_id, right_id = pair_key\n    if left_id not in selected or right_id not in selected:\n        continue\n    indicator = m.addVar(vtype=GRB.BINARY, name='')\n    m.addConstr(indicator <= selected[left_id])\n    m.addConstr(indicator <= selected[right_id])\n    m.addConstr(indicator >= selected[left_id] + selected[right_id] - 1)\n    pair_active[pair_key] = indicator\n# Read pair_active[pair_key]; do not overwrite pair_active with an endpoint string.\n# Form symbolic comparisons only after computing numeric parameter branches.\nfor item in item_ids:\n    allowed = int(eligibility[item])\n    m.addConstr(selected[item] <= allowed)\n# Only when the Query requests signed ledger totals: AFTER its scope/revision/DELETE/\n# dedup rules, accumulate every effective row by the complete required key.\ncapacity = {}\nfor record in effective_ledger_records:\n    values = record['values']\n    resource = values['resource']\n    capacity[resource] = capacity.get(resource, 0) + float(values['amount'])\n# Do not overwrite a total with the last reservation entry or discard its negative sign.\nUse exact source-family names from logical_records, not example filenames. Keep every\nreturned fragment; source_position is returned-view order, not pre-extraction CSV order.\n"

def _legacy_records_from_observation(legacy_observation):
    """Parse legacy evidence only when its structure is unambiguous.

    New runs use planned JSON payloads. This compatibility parser accepts JSON
    arrays/objects, JSON Lines, CSV, and Markdown tables, but rejects ragged or
    otherwise ambiguous input instead of silently constructing malformed records.
    """
    import csv, io

    def normalize_row(row, source):
        if not isinstance(row, dict):
            raise ValueError("Legacy observation rows must be JSON objects or mappings")
        values = row.get("values", row)
        if not isinstance(values, dict) or not values:
            raise ValueError("Legacy observation row has no mapping of values")
        if any(key is None or not str(key).strip() for key in values):
            raise ValueError("Legacy observation contains a missing CSV column name")
        return {"source": source, "values": {str(key).strip(): value for key, value in values.items()}}

    text = str(legacy_observation or "").strip()
    if not text:
        raise ValueError("Legacy observation is empty")

    # Prefer a complete JSON payload, including JSON arrays.
    try:
        decoded = json.loads(text)
    except json.JSONDecodeError:
        decoded = None
    if decoded is not None:
        if isinstance(decoded, dict) and isinstance(decoded.get("tables"), list):
            records = []
            for table in decoded["tables"]:
                if not isinstance(table, dict):
                    raise ValueError("Legacy observation table must be an object")
                source = str(table.get("source") or table.get("file_name") or "")
                rows = table.get("records")
                if not isinstance(rows, list):
                    raise ValueError("Legacy observation table records must be a list")
                records.extend(normalize_row(row, source) for row in rows)
            if records:
                return records
            raise ValueError("Legacy observation JSON contains no records")
        rows = decoded if isinstance(decoded, list) else [decoded]
        if all(isinstance(row, dict) for row in rows):
            return [normalize_row(row, "") for row in rows]
        raise ValueError("Legacy observation JSON must contain record objects")

    # Split optional source headings, then parse each section strictly.
    sections = re.split(r"(?m)^(?:Retrieved data from )?([\w.-]+\.csv)(?: \(in source order\):)?[ \t]*$", text)
    sections = ["", sections[0], *sections[1:]]
    records = []
    for source, section in zip(sections[::2], sections[1::2]):
        lines = [line.strip() for line in section.splitlines() if line.strip()]
        if not lines:
            continue
        if lines[0].startswith("```"):
            lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines.pop()
        if not lines:
            continue

        # A source heading may precede either a JSON array/object or JSON Lines.
        section_text = "\n".join(lines)
        try:
            section_decoded = json.loads(section_text)
        except json.JSONDecodeError:
            section_decoded = None
        if section_decoded is not None:
            section_rows = section_decoded if isinstance(section_decoded, list) else [section_decoded]
            if not all(isinstance(row, dict) for row in section_rows):
                raise ValueError("Legacy observation JSON section must contain record objects")
            records.extend(normalize_row(row, source) for row in section_rows)
            continue
        if all(line.lstrip().startswith("{") for line in lines):
            try:
                records.extend(normalize_row(json.loads(line), source) for line in lines)
            except json.JSONDecodeError as exc:
                raise ValueError("Legacy JSON Lines observation is malformed") from exc
            continue

        # Markdown table: require a separator row and equal column counts.
        if "|" in lines[0] and len(lines) >= 2 and "-" in lines[1]:
            split_row = lambda line: [part.strip() for part in line.strip().strip("|").split("|")]
            headers, separator = split_row(lines[0]), split_row(lines[1])
            if not headers or len(headers) != len(separator) or any(not h for h in headers):
                raise ValueError("Malformed Markdown table header")
            if any(not re.fullmatch(r":?-{1,}:?", cell) for cell in separator):
                raise ValueError("Malformed Markdown table separator")
            for line in lines[2:]:
                values = split_row(line)
                if len(values) != len(headers):
                    raise ValueError("Ragged Markdown table row")
                records.append({"source": source, "values": dict(zip(headers, values))})
            continue

        reader = csv.DictReader(io.StringIO("\n".join(lines)), skipinitialspace=True)
        if not reader.fieldnames or any(field is None or not field.strip() for field in reader.fieldnames):
            raise ValueError("Legacy CSV observation has invalid column names")
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError("Legacy CSV observation has a ragged row")
            records.append(normalize_row(row, source))
    if not records:
        raise ValueError("Legacy observation format is unsupported or contains no records")
    return records


# ============================================================
# Code generation: shared RAG examples and one prompt builder
# ============================================================

ORIGINAL_CODE_PROMPT = """
You are an expert in mathematical optimization and Python programming.
Write executable Python code that solves the provided mathematical optimization
model with Gurobi, including every decision variable, objective, and constraint.
Preserve the formulation's variable domains. Return code only, without explanations
or Markdown fences.

Mathematical Optimization Model:
{output}
"""

CSV_ROUTE_HINT = {
    "FLP": "Preserve facility-opening and assignment indices and their linking constraints.",
    "AP": "Preserve assignment indices and the matching constraints in the formulation.",
    "TP": "Preserve source-destination axes and supply-demand balance; never assume a square matrix.",
    "RA": """Preserve resource dimensions and capacity constraints. Keep independent
resource-by-activity allocations distinct from global activities consuming multiple
resource dimensions. Index independent allocations by resource, and use each
resource's own decisions or consumption coefficients in its capacity constraint.
Never reuse a global allocation expression against every independent warehouse's
capacity. Preserve all supplied capacities, coefficients, and business identifiers.
Implement the variable domains explicitly given in the current formulation;
do not reinterpret them from the assigned route or a coding example.""",
    "NRM": "Preserve product/resource indices and dimension-specific capacity coefficients.",
    "Others": "Follow the formulation exactly; do not introduce unsupported assumptions.",
}

PLANNED_CODE_INSTRUCTIONS = """
Use the formulation for model structure and Data Mapping; use CSVQA_DATA for all data.
CSVQA_DATA at execution contains every original returned record and coefficient. The prompt may show an explicitly marked source profile: its records are samples and all _prompt_* fields are absent at runtime. Iterate complete runtime records, never copy sample dimensions or access prompt-only fields. Do not define
or overwrite it, copy its rows into literals, or read external files.
Select tables by the exact table_id in Data Mapping; roles may repeat.
Derive dimensions and identifiers from CSVQA_DATA["tables"][i]["records"], reading
fields through record["values"][column_name]. Preserve table mapping, record order,
identifiers, and matrix axes. Do not read CSVQA_DATA["plan"].
Without a continuous pre-horizon value, start change/ramp constraints at the second
period; an initial binary state does not supply a continuous period-zero value.
Optional row_id_mapping/column_id_mapping in a declared matrix relationship are
dictionaries from raw matrix label strings to entity ID strings. Apply them only when
present; otherwise use exact raw IDs. Obtain a row label from record['values'][row_id_column],
never from the record dictionary itself. Preserve every coefficient and both original IDs.
Derive complete index sets from current records; an example or a copied formulation's
literal entity list must not narrow them without an explicit original-query restriction.
Records are already structured dictionaries; do not reparse the payload as CSV text.
Only parse individual source fields when those fields themselves contain structured text.
"""

LEGACY_CODE_INSTRUCTIONS = """
Build a self-contained program using the concrete coefficients and identifiers in
the Mathematical Optimization Model / Retrieved Information and Original Query.
Define all required data in the code. Do not use CSVQA_DATA, external variables,
or external files.
"""

CSV_SOLVER_INSTRUCTIONS = "\nKeep business identifiers as Python dictionary keys, separate from Gurobi names.\nUse name='' for addVars/addConstrs so business keys are not expanded into solver names;\nuse only short ASCII names for individually named variables and constraints.\nUse setObjective for the original query's objective sense and full expression, including constants.\nDo not substitute an optimizer-equivalent objective or restore omitted constants only in printed output.\nNever invent, truncate, or replace required coefficients with dummy or random data.\nBefore optimization, validate coefficient dimensions and identifier coverage against decision index sets;\nraise an explicit error for missing data instead of guessing, padding, or silently dropping entries.\nFor |linear expression| <= bound, use expression <= bound and expression >= -bound;\ndo not pass a non-variable expression to addGenConstrAbs.\n\nImport gurobipy as gp and GRB from gurobipy; never import a module named gp.\nUse only Python standard-library modules, pandas, numpy, and gurobipy.\nUse gp.quicksum, never GRB.quicksum.\nGurobi addVars supports lb, ub, obj, vtype and name only; do not invent keyword arguments.\nUse a list of unique flat tuple keys with addVars(keys); do not pass a tuplelist and\nanother Cartesian index unless that extra dimension is intended. Preserve every component\nof composite business keys in dictionaries and joins. Sum repeated component rows by the\ncomplete key instead of overwriting them. Ineligible decision pairs must be constrained to\nzero or excluded; their presence in source data is not a reason to reject the entire input.\nNever use chained comparisons, Python and/or, or bitwise |/& on Gurobi constraints.\nRepresent logical conditions with explicit binary variables and indicator/linear constraints.\nUse pandas Series.str.casefold() for vectorized text and string.casefold() for one string.\n\nSet MIPGap=1e-6 before optimize().\nIf Status == GRB.OPTIMAL, print ObjVal and every variable's VarName and X.\nOtherwise print the solver status; do not report an incumbent as optimal.\n"


CSV_SOLVER_INSTRUCTIONS += '\nKeep identifiers in one consistent representation; numeric coefficients are separate.\nUse set(ids).issubset(parameter) for necessary dictionary coverage checks, separately for\neach required parameter; never invent defaults or zero-fill missing required coefficients.\nNever mix set subtraction and union in a missing-data expression.\n'

def retrieve_csv_code_example(route, query):
    """Render code references from the same structured examples used for modeling."""
    examples = []
    for row in retrieve_rag_examples(normalize_route(route), query, k=2):
        if row["Code"].strip():
            examples.append(
                f"Example {len(examples) + 1}:\n"
                f"Problem:\n{row['prompt']}\n\n"
                f"Formulation:\n{row['Label']}\n\n"
                f"Code:\n{row['Code']}"
            )
    return "\n\n".join(examples)


def _generate_code(output, route, original_query="", data_payload="", legacy_observation=""):
    route = normalize_route(route)
    bind_observation = route == "RA" and bool(legacy_observation) and not data_payload
    parts = [ORIGINAL_CODE_PROMPT.format(output=output)]
    if original_query:
        parts.append(f"Original Query:\n{original_query}")
    if data_payload:
        shown_data = csvqa_prompt_view(data_payload)
        if "logical_records" in json.loads(data_payload):
            full_data = json.loads(data_payload)
            prompt_data = json.loads(shown_data)
            if "_prompt_view" in prompt_data:
                prompt_data["logical_records"] = {
                    family: rows[:1] + (rows[-1:] if len(rows) > 1 else [])
                    for family, rows in full_data["logical_records"].items()}
                prompt_data["_prompt_view"]["logical_sample_policy"] = "first and last returned record per family, not effective-record selection; complete runtime records unchanged"
                shown_data = json.dumps(prompt_data, ensure_ascii=False, sort_keys=True)
        title = "Complete CSVQA_DATA" if shown_data == data_payload else "CSVQA_DATA source profile (all runtime records are complete)"
        instructions = PLANNED_CODE_INSTRUCTIONS
        if "logical_table_index" in json.loads(data_payload):
            instructions = instructions.replace('Select tables by the exact table_id in Data Mapping; roles may repeat.\nDerive dimensions and identifiers from CSVQA_DATA["tables"][i]["records"], reading\nfields through record["values"][column_name]. Preserve table mapping, record order,\nidentifiers, and matrix axes. Do not read CSVQA_DATA["plan"].', INDEX_BINDING, 1)
            instructions = instructions.replace("CSVQA_DATA at execution contains every original returned record and coefficient.",
                "CSVQA_DATA contains the explicit prepared unit view: all returned rows remain, resource amount/unit fields are already normalized, and original-value provenance is retained. Do not convert these amounts twice.")
        if "logical_records" in json.loads(data_payload):
            prefix = INDEX_BINDING.split("Create scalar quantity/selection variables", 1)[0]
            instructions = instructions.replace(INDEX_BINDING, prefix + FOCUSED_API_PATTERNS)
        parts.extend([instructions, f"{title}:\n{shown_data}"])
    elif bind_observation:
        records = _legacy_records_from_observation(legacy_observation)
        parts.append(
            "LEGACY_RECORDS is already defined in the program as the exact records below. "
            "Use it directly: each record has 'source' and 'values'; select tables by source "
            "when nonempty, otherwise by fields in record['values']. Do not redefine it, "
            "import it, parse observation text, read files, or copy coefficients into literals. "
            "Derive dimensions and identifiers from these records. Original query determines "
            "semantics and domains; the formulation supplies structure, not guessed data. "
            "Keep explicit per-type capacity bounds; if total capacity denotes their aggregate, "
            "compute their sum rather than inventing another shared limit.\n\n"
            f"Complete LEGACY_RECORDS:\n{json.dumps(records, ensure_ascii=False)}"
        )
    else:
        parts.append(LEGACY_CODE_INSTRUCTIONS)
    current_parts = list(parts)
    native_focus = _native_codegen_focus(original_query, json.loads(data_payload)) if data_payload else None
    parts.append(CSV_ROUTE_HINT[route])

    examples = retrieve_csv_code_example(route, original_query or output)
    if examples:
        parts.extend([
            "The following examples are coding references only. Reuse indexing and "
            "Gurobi modeling patterns, but never copy their coefficients, dimensions, "
            "identifiers, or data. The current formulation and current data are authoritative.",
            examples,
        ])
    parts.extend([CSV_SOLVER_INSTRUCTIONS, MODEL_INTERFACE_INSTRUCTIONS])
    parts = ['Reference examples demonstrate methods only. Their files, entities, numerical values,\nand constraints are not inputs to the current problem. Only the Original Query and\ncurrent source observation define this instance. Use complete query-selected source\nsets, not example entity lists or profile samples. A coefficient matrix does not by\nitself introduce another decision dimension. Do not duplicate one aggregate demand\ninto separate groups unless the query or data actually distinguishes those groups.\nFor rectangular assignment matrices, retain the stated entity roles and cardinalities;\ncaption rows are not entities and row/column orientation need not match an example.\n\n<workflow_guidance>\n' + "\n\n".join(parts[len(current_parts):]) + "\n</workflow_guidance>\n<current_problem>\n" +
             "\n\n".join(current_parts) + "\n</current_problem>"]
    system = CODE_SYSTEM_INTERFACE
    if data_payload and "logical_records" in json.loads(data_payload):
        system = FOCUSED_CODE_SYSTEM
        parts = current_parts
    elif native_focus is not None:
        pattern = TREE_CODE_PATTERN if native_focus["kind"] == "rooted_connectivity_tree" else FRAGMENT_CODE_PATTERN
        system = NATIVE_CODE_SYSTEM + "\n" + pattern
        parts = ["Current native-source metadata (not extra business data):\n" +
                 json.dumps(native_focus, ensure_ascii=False, sort_keys=True)] + current_parts
    response = make_llm().invoke([SystemMessage(content=system),
                                HumanMessage(content="\n\n".join(parts))])
    code = response.content
    if bind_observation:
        code = (f"LEGACY_OBSERVATION = {legacy_observation!r}\nLEGACY_RECORDS = {records!r}\n"
                + _source_candidate(code))
    print(code)
    return code


def get_csv_code(output, route, original_query, data_payload="", legacy_observation=""):
    """Generate code with planned data or optional verbatim legacy evidence."""
    return _generate_code(output, route, original_query, data_payload, legacy_observation)


def get_code(output, selected_problem, original_query=""):
    """Generate self-contained code for a query-only formulation."""
    return _generate_code(output, selected_problem, original_query=original_query)

CSV_SOLVER_INSTRUCTIONS += '\nUse these minimal Python/Gurobi patterns when implementing the current formulation.\nThese are API examples, not extra model constraints or data for the current problem:\n\nUse the full module name gurobipy for the solver API.\nimport gurobipy\nfrom gurobipy import GRB\nmodel = gurobipy.Model()\nmodel.setParam("MIPGap", 1e-6)\n# A single index is a scalar key, not a one-element tuple.\nx = model.addVars(ids, lb=0, vtype=GRB.CONTINUOUS, name=\'\')\n# A single-index binary also uses scalar keys, read binary[i].\nbinary = model.addVars(ids, vtype=GRB.BINARY, name=\'\')\n# Multiple indices use flat tuple keys, with exactly the intended components.\ny = model.addVars(pairs, vtype=GRB.BINARY, name=\'\')\n# Add each constraint explicitly; addConstrs does not accept a dict of constraints.\nfor i in ids:\n    model.addConstr(expression_for_i <= bound_for_i)\n# Initialize parameter dictionaries once, outside their population loop.\nparameter = {}\nfor record in records:\n    values = record[\'values\']\n    parameter[str(values[id_column]).strip()] = float(values[value_column])\n# Validate each required key in an explicit loop with a bound loop variable.\nfor i in ids:\n    if i not in parameter:\n        raise ValueError(f\'Missing parameter for {i}\')\nmodel.optimize()\nif model.Status == GRB.OPTIMAL:\n    print(model.ObjVal)\n    for variable in model.getVars():\n        print(variable.VarName, variable.X)\n\nUse the current model\'s actual domains and indices, not the example names above.\nDo not import gp from gurobipy. Do not use getVarByName: retain the variable objects\nor iterate model.getVars(). Do not recreate parameter dictionaries inside row loops.\nDo not put an unbound generator after an if condition; use explicit Python loops.\nUse CSVQA_DATA directly when provided; do not define it or reproduce its records.\nReturn the complete executable code only, without printing the input payload or model.\n\nNever pass None as a Gurobi lb/ub/obj coefficient. Omit unspecified bounds or use\nnumeric GRB.INFINITY; for a declared free variable use -GRB.INFINITY below.\nWhen finite bounds are supplied, construct their complete numeric dictionaries before\naddVars. Construct complete entity/index lists before coefficient loops; an if-not-list\nappend inside a column loop retains only the first column, not the full set.\nSelect tables and columns from exact current Data Mapping, not reference filenames or\nnon-unique role text. Apply provided matrix ID mappings before parameter lookups.\n'

CSV_SOLVER_INSTRUCTIONS += 'Apply the scalar-key rule to every auxiliary variable, not just primary decisions. For a single-index variable use model.addVars([i for i in ids], ...) and variable[i]. Do not use model.addVars([(i,) for i in ids], ...) with variable[i], including MTZ order variables, opening indicators, inventory, and time-indexed variables. Use exactly the same key shape when defining and reading any decision or auxiliary variable.'

CSV_SOLVER_INSTRUCTIONS += '\nWhen the query requires reciprocal payments or equal income and expenditure, derive\nboth totals from the stated source/recipient axes before writing the equality.\nIf a[i,j] is work by source j for recipient i and p[j] is its unit rate, source i earns\np[i] * sum(a[k,i] for k != i), while recipient i spends\nsum(a[i,j] * p[j] for j != i). Self-work/self-transfers must be excluded on both sides,\nor included and cancelled on both sides. A supplied total contribution replaces the\ncorresponding full column total, not the diagonal coefficient. Never use a single\nself-entry as all income. Use this balance only when the query actually requires it.\n\nAuxiliary ordering domains must allow every required distinct position. For a tour\nwith q non-anchor nodes, use ranks 1..q (or an equivalent q-position range), not 2..q.\nOne consistent directed MTZ formulation is u[i]-u[j]+q*x[i,j] <= q-1 for distinct\nnon-anchor i,j, with 1 <= u[i] <= q. The anchor is excluded from these constraints;\ndo not require rank increases on the return edge. Match domains, big-M, index ranges\nand arc representation; do not mix pieces from different example formulations.\n'

CODE_SYSTEM_INTERFACE = 'Generate complete executable Python for the provided formulation and runtime data.\nUse the fully qualified module name gurobipy throughout the generated program.\nBegin with these imports:\nimport gurobipy\nfrom gurobipy import GRB\nCreate models with gurobipy.Model() and sums with gurobipy.quicksum(...).\nSet solver parameters with model.setParam("MIPGap", 1e-6).\nCSVQA_DATA is already supplied by the runtime. Read it directly; do not redefine it,\nserialize it into the program, or copy its records into Python literals.\nWhen the formulation indexes independent source records, use source_row as their\nstable identity. Repeated descriptive labels are not duplicate records; keep label\nlookups separate from solver variable keys.\nPass a generator to addConstrs, for example model.addConstrs((x[i] <= bound[i]\nfor i in I)), or use individual addConstr calls inside Python loops, not a list.\nKeep the provided formulation\'s indices, coefficients and domains. Return code only.\n'


## 14. Execute and assemble results

The pipeline classifies, formulates, generates code, and solves once. Query-only cases use the original Others modeling agent and the code example for the predicted class. execute_code returns the optimal objective and variable values from m or model; failures retain any model and code already generated.


In [ ]:
def _source_candidate(code):
    source = str(code).strip()
    match = re.fullmatch(r"```(?:python|py)?\s*\n(.*?)\n```", source, re.DOTALL)
    source = match.group(1) if match else source
    tree = ast.parse(source)
    # Bulk names expand business keys; keep those keys out of Gurobi display names.
    for node in ast.walk(tree):
        if (isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute)
                and node.func.attr in {"addVars", "addConstrs"}):
            node.keywords = [kw for kw in node.keywords if kw.arg != "name"]
            node.keywords.append(ast.keyword(arg="name", value=ast.Constant(value="")))
    return ast.unparse(tree)


class ModelInterfaceError(RuntimeError):
    """Generated program did not expose a live Gurobi model."""

    def __init__(self, code, detail):
        self.code = code
        super().__init__(f"{code}: {detail}")


def _resolve_generated_model(namespace):
    """Prefer m/model, otherwise accept one distinct global model; never rerun code."""
    for name in ("m", "model"):
        candidate = namespace.get(name)
        if isinstance(candidate, gp.Model):
            return candidate
    candidates = {}
    for name, value in namespace.items():
        if isinstance(value, gp.Model):
            candidates.setdefault(id(value), (value, []))[1].append(name)
    if not candidates:
        types = {name: type(namespace[name]).__name__ for name in ("m", "model") if name in namespace}
        raise ModelInterfaceError(
            "MODEL_INTERFACE_MISSING",
            f"No global Gurobi model was exposed; m/model types: {types}. "
            "A function-local model must be explicitly returned and assigned to m.",
        )
    if len(candidates) != 1:
        names = [aliases for _, aliases in candidates.values()]
        raise ModelInterfaceError("MODEL_INTERFACE_AMBIGUOUS", f"Multiple distinct global Gurobi models: {names}")
    return next(iter(candidates.values()))[0]


def execute_code(code):
    gp.setParam("TimeLimit", float(os.environ.get("LEAN_GUROBI_TIME_LIMIT", "180")))
    gp.setParam("MIPGap", 1e-6)
    gp.setParam("Threads", int(os.environ.get("LEAN_GUROBI_THREADS", "8")))
    namespace = {"__name__": "__main__"}
    exec(_source_candidate(code), namespace)
    model = _resolve_generated_model(namespace)
    try:
        status = model.Status
    except (gp.GurobiError, AttributeError) as exc:
        raise ModelInterfaceError(
            "MODEL_INTERFACE_DISPOSED", "The exposed model is unavailable; do not dispose it before result extraction."
        ) from exc
    with model:
        if status == gp.GRB.INF_OR_UNBD:
            model.Params.TimeLimit = max(0.0, model.Params.TimeLimit - model.Runtime)
            model.Params.DualReductions = 0
            model.reset()
            model.optimize()
            status = model.Status
        if status in {gp.GRB.INFEASIBLE, gp.GRB.UNBOUNDED}:
            return ("infeasible" if status == gp.GRB.INFEASIBLE else "unbounded"), []
        if status != gp.GRB.OPTIMAL:
            raise RuntimeError(f"No optimal solution; solver status: {status}")
        objective = model.ObjVal
        solution = [(v.VarName, v.X) for v in model.getVars()]
        print("Optimal objective:", objective)
        print("Optimal solution:", solution)
        return objective, solution

ROUTE_FORMULATORS = {
    "NRM": get_NRM_response, "RA": get_RA_response, "TP": get_TP_response,
    "AP": get_AP_response, "FLP": get_FLP_response,
}


def invoke_formulation(route, query, dataset_address):
    if not dataset_address:
        return {"formulation": get_others_without_CSV_response(query), "trace": {"status": "NOT_APPLICABLE"}}
    return ROUTE_FORMULATORS.get(normalize_route(route), get_Others_response)(query, dataset_address)


def case_fields(case):
    reference = case.get("label_objective")
    address = normalize_data_address(case.get("dataset_address"))
    return {
        "problem_id": str(case["problem_id"]), "query": str(case["Query"]),
        "dataset_address": address, "input_kind": "external_csv" if address else "query_only",
        "true_label": case.get("true_label"), "true_route": case.get("true_route"),
        "label_objective": parse_label_objective(reference),
    }

def classification_for_case(case):
    return invoke_classifier(case["Query"])


def execute_pipeline_case(case, *, forced_route=None):
    """Classify, formulate, generate code, and solve; retain progress if a step fails."""
    record = {**case_fields(case), "forced_route": forced_route,
              "experiment_mode": "forced" if forced_route else "automatic",
              "repair_count": 0, "retry_count": 0, "fallback_count": 0}
    HTTP_RETRY_EVENTS.clear()
    query, address = record["query"], record["dataset_address"]
    try:
        if forced_route is not None and not address:
            raise ValueError("Forced routes require CSV inputs")
        record["pipeline_stage"] = "classification" if forced_route is None else "formulation"
        classification = classification_for_case(case) if forced_route is None else {}
        label = classification.get("normalized_label")
        route = normalize_route(forced_route) if forced_route is not None else class_to_workflow_route(label)
        record.update(predicted_label=label, assigned_route=route if address else "Others")
        record["pipeline_stage"] = "formulation"
        formulation = invoke_formulation(route, query, address)
        mode = CSVQA_MODE_BY_ROUTE[route] if address else None
        payload = formulation.get("observation", "") if mode == "planned" else ""
        record.update(generated_model=formulation["formulation"], csvqa_observation=formulation.get("observation", ""), csvqa_mode=mode,
                      csvqa_status=formulation.get("trace", {}).get("status"),
                      csvqa_trace=json.dumps(formulation.get("trace", {}), ensure_ascii=False),
                      fallback_count=formulation.get("trace", {}).get("fallback_count", 0))
        model_text = formulation["formulation"]
        if (not isinstance(model_text, str) or not model_text.strip()
                or "agent stopped due to" in model_text.lower()):
            raise RuntimeError("Invalid formulation: empty model or agent stopped before producing a model")
        record["pipeline_stage"] = "code_generation"
        code = formulation.get("code")
        if code is None:
            if address:
                code = get_csv_code(
                    formulation["formulation"], route, query, data_payload=payload,
                    legacy_observation=formulation.get("observation", "")
                    if route == "RA" and mode == "legacy" else "",
                )
            else:
                code = get_code(formulation["formulation"], route, original_query=query)
        record["solve_code"] = str(code or "")
        source = _source_candidate(code)
        if payload:
            # Include NRM data so the saved program can run on its own.
            data = json.loads(payload) if isinstance(payload, str) else payload
            source = f"CSVQA_DATA = {pprint.pformat(data, sort_dicts=False, width=120)}\n{source}"
        record["solve_code"] = source
        record["pipeline_stage"] = "solve"
        objective, solution = execute_code(record["solve_code"])
        record.update(api_retry_count=len(HTTP_RETRY_EVENTS), api_retry_events=json.dumps(HTTP_RETRY_EVENTS))
        record.update(final_objective=None if isinstance(objective, str) else objective, final_solution=solution, final_ok=not isinstance(objective, str),
                      solver_status=objective.upper() if isinstance(objective, str) else "OPTIMAL",
                      record_status="completed", cache_source="computed", pipeline_stage="completed")
        return record
    except Exception as exc:
        record.update(api_retry_count=len(HTTP_RETRY_EVENTS), api_retry_events=json.dumps(HTTP_RETRY_EVENTS))
        evidence = getattr(exc, "csvqa_result", {})
        if evidence:
            record.update(csvqa_observation=evidence.get("observation", ""),
                          csvqa_trace=json.dumps(evidence.get("trace", {}), ensure_ascii=False),
                          csvqa_status=evidence.get("trace", {}).get("status"))
        record["pipeline_stage"] = getattr(exc, "pipeline_stage", record["pipeline_stage"])
        if isinstance(exc, ModelInterfaceError):
            record["pipeline_stage"] = "result_extraction"
        exc.pipeline_context = record
        raise


## 15. Test datasets and scoring

Query is required. Dataset_address, Problem Type, and the reference objective (Label-objective or Label) are optional. Missing CSV paths select query-only modeling; missing reference labels leave the corresponding accuracy blank.


In [ ]:
def normalize_data_address(value):
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return "\n".join(line.strip() for line in str(value).splitlines() if line.strip())


def resolve_dataset_address(value):
    return "\n".join(
        str((path if path.is_absolute() else PROJECT_ROOT / path).resolve())
        for path in (Path(line).expanduser() for line in normalize_data_address(value).splitlines())
    )


def parse_label_objective(value):
    """Parse a finite objective, a terminal status, or an absent reference."""
    if value is None or pd.isna(value) or not str(value).strip():
        return None
    text = str(value).replace(",", "").replace("$", "").strip()
    if text.casefold() in {"infeasible", "unbounded"}:
        return text.casefold()
    match = re.search(r"Optimal\s+objective\s*([-+\d.eE]+)", text, re.I)
    result = float(match.group(1) if match else text)
    if not math.isfinite(result):
        raise ValueError(f"Reference objective is not finite: {value!r}")
    return result


def parse_optional_gold_problem_class(value):
    """Return a supported gold label, or None when no compatible annotation exists."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return None
    if not str(value).strip():
        return None
    try:
        return normalize_problem_class(value)
    except (TypeError, ValueError):
        # A dataset may have no labels or use an unrelated external taxonomy.
        # Inference still uses invoke_classifier(query); unsupported gold labels
        # are simply excluded from classification evaluation.
        return None


def prepare_cases(test):
    """Accept a DataFrame with Query; data, objective, and gold label are optional."""
    frame = test.copy()
    if "Query" not in frame:
        raise ValueError("The test dataset must contain a Query column")
    if frame["Query"].isna().any() or frame["Query"].astype(str).str.strip().eq("").any():
        raise ValueError("Every test row must contain a non-empty Query")
    if "problem_id" not in frame:
        frame["problem_id"] = [f"OR-{index + 1:03d}" if isinstance(index, (int, np.integer)) else f"OR-{position:03d}"
                               for position, index in enumerate(frame.index, 1)]
    frame["problem_id"] = frame["problem_id"].astype(str)
    addresses = frame.get("dataset_address", frame.get("Dataset_address", pd.Series("", index=frame.index)))
    frame["dataset_address"] = addresses.map(resolve_dataset_address)
    labels = frame.get("true_label", frame.get("Gold Problem Type", frame.get(
        "gold_problem_type", frame.get("Problem Type", pd.Series(None, index=frame.index, dtype=object))
    )))
    frame["true_label"] = labels.map(parse_optional_gold_problem_class)
    frame["true_route"] = frame["true_label"].map(lambda label: class_to_workflow_route(label) if label else None)
    objective_columns = {str(column).strip().casefold(): column for column in frame.columns}
    objectives = next((frame[objective_columns[name]] for name in ("label_objective", "label-objective", "label", "lable")
                       if name in objective_columns), pd.Series(None, index=frame.index, dtype=object))
    frame["label_objective"] = objectives.map(parse_label_objective)
    return frame


def load_benchmark(path=BENCHMARK_PATH):
    return prepare_cases(read_csv_compat(path))


def objective_is_correct(actual, expected, *, rel_tol=1e-4, abs_tol=1e-4):
    try:
        actual, expected = float(actual), float(expected)
        return math.isfinite(actual) and math.isfinite(expected) and math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol)
    except (TypeError, ValueError, OverflowError):
        return False


def finalize_record(record, *, rel_tol=1e-4, abs_tol=1e-4):
    result = dict(record)
    result["classification_correct"] = (
        result.get("predicted_label") == result["true_label"]
        if result.get("true_label") and result.get("experiment_mode") == "automatic" else None
    )
    reference = parse_label_objective(result.get("label_objective"))
    if reference is None:
        result["solution_correct"] = None
    elif isinstance(reference, str):
        result["solution_correct"] = (
            result.get("record_status") == "completed" and result.get("solver_status") == reference.upper()
        )
    else:
        result["solution_correct"] = bool(
            result.get("final_ok") and objective_is_correct(result.get("final_objective"), reference, rel_tol=rel_tol, abs_tol=abs_tol)
        )
    return result

## 16. Save and reuse results

Each experiment has a small results.csv for Excel and one folder per case containing model.md, solve.py, and solution.json. CSV data overviews are saved as data_overview.md. Variables are stored as name/value pairs so repeated solver display names remain distinct. Completed rows with matching code and data fingerprints can be reused.


In [ ]:
def _write_text_atomic(path, text, encoding):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding=encoding, dir=path.parent, prefix=f".{path.name}.",
            suffix=".tmp", delete=False,
        ) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
            temporary = Path(handle.name)
        temporary.replace(path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


CSV_VALUE_READERS = {
    **dict.fromkeys(("final_ok", "classification_correct", "solution_correct"), lambda value: value == "True"),
    **dict.fromkeys(("final_objective",), float),
    "label_objective": parse_label_objective,
}
_RESULT_LOCK = RLock()


def load_records(csv_path):
    path = Path(csv_path)
    if not path.exists():
        return []
    rows = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig").to_dict("records")
    return [{key: CSV_VALUE_READERS.get(key, str)(value) if value else None
             for key, value in row.items()} for row in rows]


def record_key(record):
    return (str(record["problem_id"]), record.get("query", ""),
            normalize_data_address(record.get("dataset_address")), record.get("forced_route") or "AUTO")


RECORD_ARTIFACT_FILES = {"generated_model": "model.md", "solve_code": "solve.py", "csvqa_observation": "data_overview.md", "csvqa_trace": "csvqa_trace.json"}


def _sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def materialize_record(csv_path, record):
    """Load a saved record's artifacts so save_record can copy them to another result folder."""
    root, row = Path(csv_path).parent, dict(record)
    for field in RECORD_ARTIFACT_FILES:
        relative = row.pop(field + "_path", None)
        expected_hash = row.pop(field + "_sha256", None)
        artifact = root / relative if relative else None
        if relative is None and record.get("record_status") == "error":
            continue
        if artifact is None or not artifact.is_file() or not expected_hash:
            return None
        try:
            raw = artifact.read_bytes()
            if _sha256_bytes(raw) != expected_hash:
                return None
            row[field] = raw.decode("utf-8")
        except (OSError, UnicodeError):
            return None
    relative = row.pop("solution_path", None)
    expected_hash = row.pop("solution_sha256", None)
    artifact = root / relative if relative else None
    if artifact is None or not artifact.is_file() or not expected_hash:
        return None
    try:
        raw = artifact.read_bytes()
        if _sha256_bytes(raw) != expected_hash:
            return None
        solution = json.loads(raw.decode("utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError):
        return None
    if not isinstance(solution, dict) or "variables" not in solution:
        return None
    row["final_solution"] = solution["variables"]
    return row

def reusable_record(csv_path, record, fingerprint):
    """Reuse every recorded result, including failures."""
    if (not record or record.get("record_status") not in {"completed", "error"}
            or record.get("cache_fingerprint") != fingerprint):
        return None
    return materialize_record(csv_path, record)


def save_record(csv_path, record):
    """Save readable case files, then atomically update the small results CSV."""
    path = Path(csv_path)
    # Percent-encoding preserves distinct IDs without treating them as paths.
    identity = json.dumps(record_key(record), ensure_ascii=False, separators=(",", ":"), default=str)
    artifact_id = hashlib.sha256(identity.encode("utf-8")).hexdigest()[:16]
    folder = (path.parent / (path.stem + "_cases") / (record.get("forced_route") or "AUTO")
              / quote(str(record["problem_id"]), safe="").replace(".", "%2E") / artifact_id)
    folder.mkdir(parents=True, exist_ok=True)
    row = dict(record)
    with _RESULT_LOCK, path.with_suffix(".csv.lock").open("a+") as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        for field, filename in RECORD_ARTIFACT_FILES.items():
            content = row.pop(field, None)
            if content is not None:
                content = str(content)
                _write_text_atomic(folder / filename, content, "utf-8")
                row[field + "_path"] = str((folder / filename).relative_to(path.parent))
                row[field + "_sha256"] = _sha256_bytes(content.encode("utf-8"))
        if "final_solution" in row:
            solution = {"status": row.get("solver_status") or ("OPTIMAL" if row.get("final_ok") else "ERROR"),
                        "objective": row.get("final_objective"), "variables": row.pop("final_solution"),
                        "error": row.get("execution_error")}
            solution_text = json.dumps(solution, ensure_ascii=False, indent=2, allow_nan=False)
            _write_text_atomic(folder / "solution.json", solution_text, "utf-8")
            row["solution_path"] = str((folder / "solution.json").relative_to(path.parent))
            row["solution_sha256"] = _sha256_bytes(solution_text.encode("utf-8"))
        records = [old for old in load_records(path) if record_key(old) != record_key(row)] + [row]
        _write_text_atomic(path, pd.DataFrame(records).to_csv(index=False), "utf-8-sig")
    return records


def _source_fingerprint():
    if not NOTEBOOK_PATH.is_file():
        raise FileNotFoundError(
            f"Configured notebook for cache fingerprint does not exist: {NOTEBOOK_PATH}. "
            "Set LEAN_LLM_OPT_NOTEBOOK to the notebook being executed."
        )
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    source = "\n".join("".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and "experiment" not in cell.get("metadata", {}).get("tags", []))
    digest = hashlib.sha256((CACHE_SCHEMA_VERSION + "\n" + str(NOTEBOOK_PATH) + "\n" + source).encode())
    for path in sorted((PROJECT_ROOT / "Large_Scale_Or_Files").rglob("*.csv")):
        digest.update(str(path.relative_to(PROJECT_ROOT)).encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def case_fingerprint(case, route=None, *, source_fingerprint=None):
    fields = {"cache_schema": CACHE_SCHEMA_VERSION,
              "source": source_fingerprint if source_fingerprint is not None else _source_fingerprint(),
              "model": MODEL_SNAPSHOT,
              "embedding_model": EMBEDDING_MODEL, "modes": CSVQA_MODE_BY_ROUTE,
              "nrm_retry_on_truncation": NRM_RETRY_ON_TRUNCATION,
              "query": str(case["Query"]), "forced_route": route}
    digest = hashlib.sha256(json.dumps(fields, sort_keys=True).encode())
    for raw in normalize_data_address(case.get("dataset_address")).splitlines():
        path = Path(raw)
        digest.update(str(path).encode())
        digest.update(path.read_bytes() if path.is_file() else b"MISSING_FILE")
    return digest.hexdigest()


def error_record(case, exc, forced_route=None):
    return {
        **case_fields(case), "assigned_route": forced_route, "predicted_label": None,
        **getattr(exc, "pipeline_context", {}),
        "experiment_mode": "forced" if forced_route else "automatic", "forced_route": forced_route,
        "final_ok": False, "final_objective": None, "final_solution": None,
        "execution_error": f"{type(exc).__name__}: {exc}", "execution_error_type": type(exc).__name__,
        "execution_error_code": getattr(exc, "code", None),
        "record_status": "error", "cache_source": "computed",
        "generated_model": getattr(exc, "pipeline_context", {}).get("generated_model", ""),
        "solve_code": getattr(exc, "pipeline_context", {}).get("solve_code", ""),
        "csvqa_observation": getattr(exc, "pipeline_context", {}).get("csvqa_observation", ""),
        "csvqa_trace": getattr(exc, "pipeline_context", {}).get("csvqa_trace", "{}"),
        "repair_count": 0, "retry_count": 0,
        "fallback_count": getattr(exc, "pipeline_context", {}).get("fallback_count", 0),
    }


## 17. Shared test runner

Run each case once and save one folder per case. Recorded successes and failures are reused when the notebook is run again on the same output directory.


In [ ]:
def run_test(test, *, output_csv=RESULTS_DIR / "results.csv", routes=None, reuse_auto_csv=None,
             reuse_completed=True, continue_on_error=True, rel_tol=1e-4, abs_tol=1e-4):
    """Evaluate each case once; resume recorded successes and failures."""
    frame = load_benchmark(test) if isinstance(test, (str, Path)) else prepare_cases(test)
    selected_routes = [None] if routes is None else [normalize_route(route) for route in routes]
    if not selected_routes or len(selected_routes) != len(set(selected_routes)):
        raise ValueError("Provide a non-empty list of distinct routes")
    if routes is not None and frame["dataset_address"].eq("").any():
        raise ValueError("Forced-route tests require case CSV inputs; use routes=None for query-only tests")
    existing = load_records(output_csv)
    if existing and not reuse_completed:
        raise ValueError("Use a new round directory for a fresh run; historical attempts cannot be overwritten")
    cached = {record_key(row): row for row in existing}
    auto_cached = ({record_key(row): row for row in load_records(reuse_auto_csv)}
                   if reuse_completed and reuse_auto_csv else {})
    # Fingerprint every run before saving records so a first run is reusable later.
    source = _source_fingerprint()
    results, api_checked = [], False
    total = len(frame) * len(selected_routes)
    for case in frame.to_dict("records"):
        for route in selected_routes:
            key = (case["problem_id"], str(case["Query"]), case["dataset_address"], route or "AUTO")
            fingerprint = case_fingerprint(case, route, source_fingerprint=source)
            if key in cached and cached[key].get("cache_fingerprint") != fingerprint:
                raise ValueError("Existing attempts use different code/data; choose a new version directory")
            record = reusable_record(output_csv, cached.get(key), fingerprint)
            if key in cached and record is None:
                raise ValueError("Recorded attempt has missing/corrupt artifacts; do not rerun it")
            if record is not None:
                record["cache_source"] = "csv"
            elif route is not None and route == case.get("true_route"):
                automatic = auto_cached.get(key[:-1] + ("AUTO",))
                if automatic and automatic.get("assigned_route") == route:
                    auto_fingerprint = case_fingerprint(case, source_fingerprint=source)
                    record = reusable_record(reuse_auto_csv, automatic, auto_fingerprint)
                    if record is not None:
                        record["cache_source"] = "oracle_auto_reuse"
            if record is not None:
                record.update(**case_fields(case), forced_route=route,
                              experiment_mode="forced" if route else "automatic")
            else:
                # A missing credential is a run configuration error, not 101 case failures.
                if not api_checked:
                    require_model_backend()
                    api_checked = True
                try:
                    record = execute_pipeline_case(case, forced_route=route)
                except Exception as exc:
                    record = error_record(case, exc, route)
                    print(f"{case['problem_id']} / {route or 'AUTO'} "
                          f"failed at {record.get('pipeline_stage', 'setup')}: {record['execution_error']}")
                    if not continue_on_error:
                        record = finalize_record(record, rel_tol=rel_tol, abs_tol=abs_tol)
                        record["cache_fingerprint"] = fingerprint
                        save_record(output_csv, record)
                        raise
            record = finalize_record(record, rel_tol=rel_tol, abs_tol=abs_tol)
            record["cache_fingerprint"] = fingerprint
            record["base_notebook_sha256"] = globals().get("BASE_NOTEBOOK_SHA256", hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest())
            save_record(output_csv, record)
            results.append(record)
            print(f"[{len(results)}/{total}] {case['problem_id']} / {route or 'AUTO'}: "
                  f"{record['record_status']} [{record.get('cache_source')}]")
    return results


def evaluation_summary(records):
    frame = pd.DataFrame(records)
    if frame.empty:
        return pd.DataFrame(columns=["metric", "correct", "total", "accuracy"])
    rows = []
    if frame["experiment_mode"].eq("forced").all():
        oracle = frame.loc[
            frame["forced_route"].eq(frame["true_route"])
            & frame["solution_correct"].notna()
        ].drop_duplicates("problem_id")
        for label, correct, total in (
                ("Oracle routing accuracy", int(oracle["solution_correct"].eq(True).sum()), oracle["problem_id"].nunique()),
                ("Solved accuracy", int(frame["final_ok"].eq(True).sum()), len(frame))):
            rows.append({"metric": label, "correct": correct, "total": total,
                         "accuracy": correct / total if total else None})
        return pd.DataFrame(rows)
    for field, label in (("final_ok", "Solved"), ("classification_correct", "Classification"), ("solution_correct", "Objective match")):
        values = frame[field].dropna() if field in frame else pd.Series(dtype=bool)
        total, correct = len(values), int(values.eq(True).sum())
        rows.append({"metric": label, "correct": correct, "total": total,
                     "accuracy": correct / total if total else None})
    return pd.DataFrame(rows)


def classification_confusion(records):
    frame = pd.DataFrame(records)
    frame = frame.loc[frame["true_label"].notna() & frame["experiment_mode"].eq("automatic")]
    predicted = frame["predicted_label"].fillna("Unclassified")
    columns = CLASS_LABELS + (["Unclassified"] if predicted.eq("Unclassified").any() else [])
    return pd.crosstab(frame["true_label"], predicted).reindex(index=CLASS_LABELS, columns=columns, fill_value=0)


def accuracy_by_class(records):
    frame = pd.DataFrame(records)
    grouped = frame.groupby("true_label").agg(
        total=("problem_id", "size"), classification_correct=("classification_correct", "sum"),
        classification_scored=("classification_correct", "count"), solution_correct=("solution_correct", "sum"),
        solution_scored=("solution_correct", "count"))
    for metric in ("classification", "solution"):
        grouped[metric + "_accuracy"] = grouped[metric + "_correct"] / grouped[metric + "_scored"].replace(0, np.nan)
    return grouped.reindex(CLASS_LABELS).rename_axis("true_label")


def forced_route_matrix(records):
    frame = pd.DataFrame(records)
    grouped = frame.groupby(["true_route", "assigned_route"])["solution_correct"].agg(correct="sum", total="count")
    formatted = grouped.apply(lambda row: f"{int(row.correct)}/{int(row.total)} ({row.correct / row.total:.2%})" if row.total else "Not scored", axis=1)
    return formatted.unstack("assigned_route").reindex(index=WORKFLOW_ROUTES, columns=WORKFLOW_ROUTES)


def accuracy_by_route(records):
    grouped = pd.DataFrame(records).groupby("assigned_route")["solution_correct"].agg(correct="sum", total="count")
    grouped["accuracy"] = grouped["correct"] / grouped["total"].replace(0, np.nan)
    return grouped.reindex(WORKFLOW_ROUTES)


def report_results(records, output_dir):
    """Display and save the tables appropriate to the experiment."""
    tables = {"summary": evaluation_summary(records)}
    if records:
        if all(row.get("experiment_mode") == "automatic" for row in records):
            tables.update(classification_confusion=classification_confusion(records), accuracy_by_class=accuracy_by_class(records))
        else:
            tables.update(route_matrix=forced_route_matrix(records), accuracy_by_route=accuracy_by_route(records))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        print(name.replace("_", " ").title())
        shown = table.copy()
        for column in shown.columns:
            if str(column).endswith("accuracy"):
                shown[column] = shown[column].map(lambda value: f"{value:.2%}" if pd.notna(value) else "Not scored")
        display(shown)
        table.to_csv(output_dir / f"{name}.csv", index=table.index.name is not None, encoding="utf-8-sig")
    return tables


## 18. Component setting: RAG Only

In [ ]:
def _no_examples(*args, **kwargs):
    return ""


# RAG Only: no worked formulation or code examples; the data-handling tools are unchanged.
build_formulation_examples = _no_examples
retrieve_csv_code_example = _no_examples


## 19. Run Large-scale-or (101 cases)

Enable this cell to run the 101 instances with this component setting.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    ablation_results = run_test(BENCHMARK_PATH, output_csv=RESULTS_DIR / "automatic/results.csv")
    report_results(ablation_results, RESULTS_DIR / "automatic")
